# 25 Netzkarte mit Monitorstatus

- Kopie von Netz-Notebook 16 (Leitungsverlauf, HAST, Zähler, Kanten) plus Status des Kennlinien-Monitors. Notebook 16 und `dashboard_export.py` bleiben unverändert.
- Pfade auf dieses Repo (`data/raw`, `data/processed/real_meters`, `data/manual/`), Ausgaben nach `outputs/netz/`.
- Statt `%run -i dashboard_export.py` läuft `dashboard_monitor.py`: Färbungen nach Priorität und frischem Alarmbeginn, Monitorzeilen im Hausfenster, Netzereignis-Hinweis, Knopf "Ganze Analyse anzeigen". Ohne `mapclassify` bleiben die `explore()`-Karten leer.
- Monitordaten aus Notebook 24. Ausgaben der kopierten Zellen und `outputs/netz/netzkarte_monitor.html` enthalten Adressen und bleiben intern.

In [ ]:
import os
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
NETZ_AUSGABE = ROOT / "outputs" / "netz"
NETZ_AUSGABE.mkdir(parents=True, exist_ok=True)
os.chdir(NETZ_AUSGABE)

for datei in ("monitor_status.csv", "monitor_alarme.csv", "netzereignistage.csv"):
    if not (ROOT / "outputs" / datei).exists():
        raise FileNotFoundError(f"outputs/{datei} fehlt, zuerst Notebook 24 ausführen")
KOORDINATEN = ROOT / "data" / "manual" / "adressen_koordinaten.ods"
if not KOORDINATEN.exists():
    raise FileNotFoundError("data/manual/adressen_koordinaten.ods fehlt (Handliste aus Notebook 16)")

try:
    import mapclassify
except ImportError:
    import folium
    import geopandas as gpd

    def _ohne_karte(self, *args, m=None, **kwargs):
        return m if m is not None else folium.Map()

    gpd.GeoDataFrame.explore = _ohne_karte
    gpd.GeoSeries.explore = _ohne_karte
    print("mapclassify fehlt: die explore-Karten aus Notebook 16 bleiben leer")
print("Ausgaben nach", NETZ_AUSGABE.relative_to(ROOT))

## Kopie von Notebook 16

- Zellen 1 bis 55 aus Netz-Notebook 16, Ausgaben wegen Adressen gelöscht.

# Netz: Leitungsverlauf, HAST, Zähler und Kanten

Ablauf:
1. Leitungsverlauf aus den Shapefiles und der KML einlesen, Rohrdaten auf die KML übertragen
2. HAST-Kandidaten aus freien Leitungsenden bestimmen
3. Zähler und Adressen aus `Zuordnung.xlsx` zusammenführen
4. Stammliste mit `Nodes_Edges.ods` abgleichen, Kanten als gerichteten Graphen auswerten
5. Positionen der Adressen laden und Adressen den Rohrenden zuordnen
6. Zähler mit Zeitreihen (Menge X) auf Kanten und Karte bringen

Abhängigkeiten: `requirements.txt`, zusätzlich `geopy` (nur für die optionale Nominatim-Suche).
Das Notebook erwartet als Arbeitsverzeichnis den Ordner `notebooks/`; Ergebnisdateien werden dort abgelegt.

In [ ]:
import re
import difflib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import folium
import pyogrio
import shapely
from folium.plugins import PolyLineTextPath
from shapely import force_2d
from shapely.geometry import LineString, Point
from shapely.ops import substring, unary_union
from io import StringIO
import matplotlib.dates as mdates
import folium, branca

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 200)

DATA_RAW    = ROOT / "data" / "raw"
SHP_ORDNER  = DATA_RAW   #hier liegen die Shapefiles direkt in data/raw
KML_PFAD    = DATA_RAW / "Leitungsverlauf.kml"
NODES_EDGES = DATA_RAW / "Nodes_Edges.ods"
ZUORDNUNG   = DATA_RAW / "Zuordnung.xlsx"
REAL_METERS = ROOT / "data" / "processed" / "real_meters"

CRS_UTM = 25832                                          # ETRS89 / UTM 32N, wie die Shapefiles
CSV = dict(sep=";", index=False, encoding="utf-8-sig")   # Einstellungen für alle CSV-Exporte

## Hilfsfunktionen

In [ ]:
UMLAUTE = {"ß": "ss", "ä": "ae", "ö": "oe", "ü": "ue"}


def spaltenname(c):
    return re.sub(r"\s+", " ", str(c)).strip().lower()


def finde_spalte(df, name=None, stichworte=()):
    """Spalte über den genauen Namen finden, sonst über das erste passende Stichwort."""
    cols = {spaltenname(c): c for c in df.columns}
    if name and spaltenname(name) in cols:
        return cols[spaltenname(name)]
    return next((c for c in df.columns if any(s in spaltenname(c) for s in stichworte)), None)


def norm_adresse(s):
    """Vergleichsschlüssel für Adressen, z.B. 'Hauptstr. 12 a' -> 'hauptstrasse 12a'."""
    s = str(s).strip().lower()
    for a, b in UMLAUTE.items():
        s = s.replace(a, b)
    s = re.sub(r"str\.", "strasse", s)
    s = re.sub(r"[^\w\s-]", " ", s)
    s = re.sub(r"str(?=\s|$)", "strasse", s)
    s = re.sub(r"\s+", " ", s).strip()
    return re.sub(r"(\d+)\s+([a-z])\b", r"\1\2", s)


def adress_key(s):
    """norm_adresse für Tabellenzellen, leere Zellen ergeben None."""
    return norm_adresse(s) if isinstance(s, str) and s.strip() else None


def nur_strasse(s):
    """Adressschlüssel ohne Hausnummer (auch ohne Bereiche wie '12-14')."""
    return norm_adresse(re.sub(r"\s*\d+\s*[a-zA-Z]?(\s*[-/–]\s*\d+\s*[a-zA-Z]?)?\s*$", "", str(s).strip()))


def teile_adresse(a):
    """'Hauptstraße 12 a' -> ('Hauptstraße', '12a')"""
    m = re.match(r"^(.*?\D)\s*(\d+\s*[a-zA-Z]?)\s*$", re.sub(r"\s+", " ", str(a)).strip())
    return (m.group(1).strip(" ,"), m.group(2).replace(" ", "")) if m else (str(a), "")


def norm_z(s):
    """Zählernummer: nur Ziffern, ohne führende Nullen."""
    return re.sub(r"\D", "", str(s)).lstrip("0")


def zahl_sauber(x):
    """Zahlen-Spalte als Text ohne angehängtes '.0'."""
    return x.fillna("").str.strip().str.replace(r"\.0$", "", regex=True)


def dn_klasse(dn, dn_klein):
    if pd.isna(dn):
        return "unbekannt"
    return f"klein (≤ DN {dn_klein})" if dn <= dn_klein else "groß"


def parse_koordinate(s):
    """'49.81, 10.34' oder '49,81234 10,34567' -> (lat, lon), sonst (None, None)."""
    if not isinstance(s, str) or not s.strip():
        return (None, None)
    teile = [x.strip() for x in s.strip().replace(";", ",").split(",")]
    try:
        if len(teile) == 2:
            return float(teile[0]), float(teile[1])
        zahlen = re.findall(r"-?\d+(?:[.,]\d+)?", s)
        if len(zahlen) == 2:
            return tuple(float(z.replace(",", ".")) for z in zahlen)
    except ValueError:
        pass
    return (None, None)


def punkte_gdf(df):
    """GeoDataFrame (WGS84) aus den Spalten lat/lon."""
    return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=4326)


def karte_abschliessen(m, bounds, datei=None):
    """Ebenen-Auswahl, Ausschnitt auf bounds (minx, miny, maxx, maxy in WGS84), optional als HTML speichern."""
    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    if datei:
        m.save(datei)      # zusätzlich als Datei, falls die Anzeige im Notebook hängt
    return m

## 1 Leitungsverlauf aus den Shapefiles

In [ ]:
layer = {}
for shp in sorted(SHP_ORDNER.glob("*.shp")):
    gdf = gpd.read_file(shp)
    layer[shp.stem] = gdf

    print("=" * 70)
    print(shp.name)
    print("Anzahl Objekte :", len(gdf))
    print("Geometrietyp   :", list(gdf.geom_type.unique()))
    print("Koordinatensys.:", gdf.crs)
    print("Spalten        :", list(gdf.columns))
    display(gdf.head())

In [ ]:
m = layer["Leitungsverlauf_LineString"].explore(name="Linien", color="red", tooltip=True)
layer["Leitungsverlauf_Point"].explore(m=m, name="Punkte", color="blue")
layer["Leitungsverlauf_MultiPoint"].explore(m=m, name="MultiPoint", color="green")
folium.LayerControl().add_to(m)
m

Gelöschte Objekte (`DELETED`) herausfiltern, MultiLineStrings zerlegen, Längen nach Linientyp.

In [ ]:
linien = layer["Leitungsverlauf_LineString"]
print(linien["DELETED"].value_counts(dropna=False))

linien = linien[linien["DELETED"] == 0].copy()
linien = linien.explode(index_parts=False).reset_index(drop=True)
linien["laenge_m"] = linien.length

print("Linien:", len(linien), "| Gesamtlänge km:", round(linien["laenge_m"].sum() / 1000, 1))
print(linien.groupby(["LINIENTYP", "TEXT_LBL"], dropna=False)["laenge_m"].agg(["count", "sum"]).round(0))

Karte nach Linientyp, um zu prüfen, ob Typ 1 und 2 für Vor- und Rücklauf und Typ 3 für Doppelrohr stehen.

In [ ]:
linien.assign(typ=linien["LINIENTYP"].astype(str)).explore(
    column="typ", categorical=True,
    tooltip=["LINIENTYP", "TEXT_LBL", "laenge_m"],
    tiles="OpenStreetMap",
)

In [ ]:
NETZ_TYPEN = [1.0, 2.0, 3.0, 5.0]          # Einzelrohre und Doppelrohre
rohre = linien[linien["LINIENTYP"].isin(NETZ_TYPEN)].copy()
print("Rohre:", len(rohre), "|", round(rohre["laenge_m"].sum() / 1000, 1), "km")

Sind die Typ-6-Linien (Hausanschlüsse?) geschlossen, und wie weit enden sie vom Hauptnetz?

In [ ]:
hausl = linien[linien["LINIENTYP"] == 6.0].copy()
print("geschlossene Linien:", int(hausl.geometry.is_closed.sum()), "von", len(hausl))
print("Länge Median (m):", round(hausl["laenge_m"].median(), 1))

hauptnetz = unary_union(rohre.geometry)

def abstaende(g):
    a = Point(g.coords[0][:2]).distance(hauptnetz)
    b = Point(g.coords[-1][:2]).distance(hauptnetz)
    return min(a, b), max(a, b)               # nahes Ende = Seite zum Hauptnetz, fernes = Hausseite

hausl[["abstand_nah", "abstand_fern"]] = hausl.geometry.apply(lambda g: pd.Series(abstaende(g)))

bins = [-0.01, 0.5, 2, 5, 10, 25, float("inf")]
print(pd.cut(hausl["abstand_nah"], bins).value_counts().sort_index())

## 2 Leitungsverlauf aus der KML und Rohrdaten aus dem Shapefile

In [ ]:
kml_ebenen = {}
for name, _ in pyogrio.list_layers(KML_PFAD):
    gdf = gpd.read_file(KML_PFAD, layer=name)
    kml_ebenen[name] = gdf
    print("=" * 70)
    print("Ebene:", name, "| Objekte:", len(gdf), "| Typen:", list(gdf.geom_type.unique()))
    print("Spalten:", list(gdf.columns))

# Erste Ebene, nur Linien, ohne Z-Koordinate und im gleichen Koordinatensystem wie die Shapefiles
kml_linien = next(iter(kml_ebenen.values())).copy()
kml_linien = kml_linien[kml_linien.geom_type.isin(["LineString", "MultiLineString"])]
kml_linien = kml_linien.explode(index_parts=False).reset_index(drop=True)
kml_linien["geometry"] = kml_linien.geometry.apply(force_2d)
kml_linien = kml_linien.to_crs(CRS_UTM)
kml_linien["laenge_m"] = kml_linien.length

print("\nKML-Linien:", len(kml_linien), "| Länge km:", round(kml_linien["laenge_m"].sum() / 1000, 1))
print("Shapefile-Rohre:", len(rohre), "| Länge km:", round(rohre["laenge_m"].sum() / 1000, 1))
display(kml_linien.drop(columns="geometry").head(10))

In [ ]:
m = rohre.explore(color="red", name="Shapefile-Rohre", style_kwds={"weight": 4})
kml_linien.explore(m=m, color="blue", name="KML", style_kwds={"weight": 2})
folium.LayerControl(collapsed=False).add_to(m)
m

Jeder KML-Linie das Shapefile-Rohr mit der größten Überdeckung zuordnen und die Beschriftung in Material, Bauart und Dimension zerlegen.

In [ ]:
PUFFER = 2.0    # Meter Toleranz zwischen KML-Linie und Shapefile-Rohr

kml = kml_linien.copy().reset_index(drop=True)
kml["kid"] = kml.index
kml["kml_laenge_m"] = kml.length

rohr = rohre[["LINIENTYP", "TEXT_LBL", "geometry"]].reset_index(drop=True).copy()
rohr["rid"] = rohr.index
rohr_puffer = rohr.copy()
rohr_puffer["geometry"] = rohr.buffer(PUFFER)

# Pro KML-Linie das Shapefile-Rohr mit der größten Überdeckung
ov = gpd.overlay(kml[["kid", "geometry"]], rohr_puffer, how="intersection", keep_geom_type=True)
ov["deckung_m"] = ov.length
best = (ov.sort_values("deckung_m", ascending=False)
          .drop_duplicates("kid")[["kid", "LINIENTYP", "TEXT_LBL"]])
kml = kml.merge(best, on="kid", how="left")

# Anteil der KML-Linie, der auf einem Shapefile-Rohr liegt
puffer_union = rohr_puffer.geometry.union_all()
kml["deckung"] = (kml.geometry.intersection(puffer_union).length / kml["kml_laenge_m"]).clip(upper=1).round(2)

def zerlege(lbl):
    if not isinstance(lbl, str):
        return pd.Series({"material": "unbekannt", "bauart": "unbekannt", "dn": None, "mantel": None})
    l = lbl.lower()
    material = "Stahl" if "stahl" in l else ("PE-X" if "pe-x" in l else "sonstige")
    bauart = "Doppelrohr" if "doppel" in l else ("Einzelrohr" if "einzel" in l else "unbekannt")
    m = re.search(r"h-\s*(\d+)(?:\+\d+)?\s*/\s*(\d+)", l) or re.search(r"(\d+)\s*/\s*(\d+)", l)
    return pd.Series({"material": material, "bauart": bauart,
                      "dn": int(m.group(1)) if m else None, "mantel": int(m.group(2)) if m else None})

kml = kml.join(kml["TEXT_LBL"].apply(zerlege))
kml["rohrart"] = kml["material"] + " " + kml["bauart"]

print("KML-Linien:", len(kml), "| davon mit Rohrdaten:", int(kml["TEXT_LBL"].notna().sum()))
print("Deckung im Mittel:", kml["deckung"].mean().round(2))
print(kml.groupby("rohrart")["kml_laenge_m"].agg(["count", "sum"]).round(0))

# Rohre, die nur im Shapefile vorkommen
kml_puffer_union = kml.buffer(PUFFER).union_all()
nur_shp = rohr.copy()
nur_shp["geometry"] = rohr.geometry.difference(kml_puffer_union)
nur_shp = nur_shp[~nur_shp.geometry.is_empty].explode(index_parts=False)
nur_shp["laenge_m"] = nur_shp.length.round(0)
nur_shp = nur_shp[nur_shp["laenge_m"] > 3]
print("\nNur im Shapefile:", round(nur_shp["laenge_m"].sum() / 1000, 1), "km von", round(rohr.length.sum() / 1000, 1), "km")

In [ ]:
extra = [c for c in kml_linien.columns if c not in ("geometry", "laenge_m")][:4]   # KML-eigene Spalten
tooltip = ["rohrart", "TEXT_LBL", "dn", "mantel", "kml_laenge_m", "deckung"] + extra

m = kml.to_crs(4326).explore(column="rohrart", categorical=True, name="KML-Leitungen mit Rohrdaten",
                             tooltip=tooltip, style_kwds={"weight": 4})
if len(nur_shp):
    nur_shp.to_crs(4326).explore(m=m, color="gray", name="Nur im Shapefile (nicht in der KML)",
                                 tooltip=["TEXT_LBL", "laenge_m"], style_kwds={"weight": 2, "dashArray": "6 6"})
karte_abschliessen(m, kml.to_crs(4326).total_bounds, "karte_netz.html")

## 3 HAST-Kandidaten aus freien Leitungsenden

Nahe Endpunkte werden zu einem Punkt zusammengefasst. Ein Punkt gilt als freies Ende, wenn alle dort endenden Leitungen in dieselbe Richtung weglaufen
und keine andere Leitung quer durchläuft (parallele Nachbarleitungen, z.B. das Gegenstück beim Einzelrohr, zählen nicht).
Zum Vergleich wird die einfache Methode mitgezählt (kein anderes Leitungsstück im Umkreis von 0,5 m).

In [ ]:
MERGE_M  = 2.0    # Endpunkte bis zu diesem Abstand gehören zu einem Punkt
WINKEL   = 40     # alle Leitungen eines freien Endes müssen in dieselbe Richtung weglaufen (Grad)
PROBE_M  = 4.0    # so weit wird in die Leitung hineingemessen, um ihre Richtung zu bestimmen
DN_KLEIN = 50     # bis zu dieser Dimension gilt ein Ende als wahrscheinlicher Hausanschluss (Faustregel)

L = kml[kml.geom_type == "LineString"].copy()
print("Linien ohne LineString-Geometrie (übersprungen):", len(kml) - len(L))
L = L[L.length > 0.1].reset_index(drop=True)

def richtung(line, ende):
    """Einheitsvektor vom Leitungsende in die Leitung hinein."""
    s = min(PROBE_M, line.length / 2)
    if ende == 0:
        p0, p1 = line.interpolate(0), line.interpolate(s)
    else:
        p0, p1 = line.interpolate(line.length), line.interpolate(line.length - s)
    v = np.array([p1.x - p0.x, p1.y - p0.y])
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

EP = gpd.GeoDataFrame(
    [{"kid": kid, "ende": ende, "geometry": g.interpolate(0 if ende == 0 else g.length), "dir": richtung(g, ende)}
     for kid, g in zip(L["kid"], L.geometry) for ende in (0, 1)],
    crs=kml.crs)

# Nahe Endpunkte zu einem Punkt zusammenfassen
cl = EP.geometry.buffer(MERGE_M / 2).union_all()
teile = list(cl.geoms) if hasattr(cl, "geoms") else [cl]
CL = gpd.GeoDataFrame({"cid": range(len(teile))}, geometry=teile, crs=EP.crs)
EP = gpd.sjoin(EP, CL, predicate="within").drop(columns="index_right")

sidx = L.sindex

def winkel_grad(u, v):
    return np.degrees(np.arccos(np.clip(np.dot(u, v), -1, 1)))

def bewerte(grp):
    zentrum = unary_union(list(grp.geometry)).centroid
    dirs = np.vstack(grp["dir"].to_list())
    maxw = max((winkel_grad(dirs[i], dirs[j]) for i in range(len(dirs)) for j in range(i + 1, len(dirs))), default=0.0)
    hier = set(grp["kid"])
    quer = False
    for k in sidx.query(zentrum.buffer(MERGE_M), predicate="intersects"):
        if L["kid"].iloc[k] in hier:
            continue
        ln = L.geometry.iloc[k]
        pos = ln.project(zentrum)
        a, b = ln.interpolate(max(pos - 1, 0)), ln.interpolate(min(pos + 1, ln.length))
        v = np.array([b.x - a.x, b.y - a.y])
        nv = np.linalg.norm(v)
        if nv == 0:
            continue
        if np.degrees(np.arccos(min(abs(np.dot(v / nv, dirs[0])), 1))) > WINKEL:
            quer = True
            break
    return {"geometry": zentrum, "n_enden": len(grp), "kids": list(grp["kid"]),
            "winkel": round(maxw), "quer": quer, "frei": (maxw <= WINKEL) and not quer}

cluster = pd.DataFrame([{"cid": cid, **bewerte(grp)} for cid, grp in EP.groupby("cid")])
EP["neu_frei"] = EP["cid"].map(dict(zip(cluster["cid"], cluster["frei"])))
EP["alt_frei"] = EP.geometry.apply(
    lambda p: len({L["kid"].iloc[k] for k in sidx.query(p.buffer(0.5), predicate="intersects")}) <= 1)

info = kml.set_index("kid")[["rohrart", "TEXT_LBL", "dn", "mantel", "kml_laenge_m"]]
EP["rohrart"] = EP["kid"].map(info["rohrart"])
print("Freie Leitungsenden nach Rohrart, alte gegen neue Methode:")
display(EP.groupby("rohrart").agg(enden=("kid", "size"), alt_frei=("alt_frei", "sum"), neu_frei=("neu_frei", "sum")))

hast = gpd.GeoDataFrame(cluster[cluster["frei"]].drop(columns=["frei"]).reset_index(drop=True),
                        geometry="geometry", crs=EP.crs)

def nimm(kids, sp):
    """Erster bekannter Wert der Spalte sp unter den Leitungen, die an diesem Punkt enden."""
    w = info.loc[kids, sp].dropna()
    if sp == "rohrart":
        w = w[w != "unbekannt unbekannt"]
    return w.iloc[0] if len(w) else ("unbekannt" if sp == "rohrart" else None)

for sp in info.columns:
    hast[sp] = [nimm(k, sp) for k in hast["kids"]]
hast["klasse"] = hast["dn"].apply(dn_klasse, dn_klein=DN_KLEIN)
hast["hast_nr"] = hast.index + 1

print("\nFreie Enden insgesamt:", len(hast))
print(hast["rohrart"].value_counts().to_dict())
print(hast["klasse"].value_counts().to_dict())
print("Davon mit mehreren Leitungen am selben Punkt (z.B. Vor- und Rücklauf):", int((hast["n_enden"] > 1).sum()))

exp = hast.drop(columns=["geometry", "kids", "cid"]).assign(rechtswert=hast.geometry.x.round(2),
                                                           hochwert=hast.geometry.y.round(2))
exp.to_csv("hast_kandidaten.csv", **CSV)

## 4 Zähler und Adressen aus `Zuordnung.xlsx`

Pro Jahresblatt Zähler und Adresse einlesen, Adressen vereinheitlichen und Änderungen zwischen den Jahren auflisten.

In [ ]:
# Einstellungen pro Blatt. Der Blattname muss die Jahreszahl enthalten.
# Entweder "adresse" (Straße + Hausnummer in einer Spalte) ODER "strasse" + "hausnr".
# "zusatz" ist optional (z.B. eine Spalte mit "a", "b" oder "Hinterhaus").
SPALTEN = {
    "2024": {"zaehler": "Zählernummer", "adresse": "Abnahmestelle"},
    "2025": {"zaehler": "Zählernummer", "adresse": "Abnahmestelle"},
    "2026": {"zaehler": "Zählernr.", "strasse": "Straße", "hausnr": "Hausnr."},
}
IGNORIEREN = {"86792512", "41566330", "41566332", "41566331", "48090145", "41628863"}

zuord_blaetter = pd.read_excel(ZUORDNUNG, sheet_name=None, dtype=str)
for name, df in zuord_blaetter.items():
    print(f"Blatt {name!r} ({len(df)} Zeilen): {list(df.columns)}")
print()

teile = []
for name, df in zuord_blaetter.items():
    jahr = re.search(r"20\d\d", str(name))
    jahr = jahr.group() if jahr else str(name)
    cfg = SPALTEN.get(jahr, {})

    z = finde_spalte(df, cfg.get("zaehler"), ["zähler", "zaehler"])
    if "strasse" in cfg:
        s = finde_spalte(df, cfg.get("strasse"), ["stra"])
        h = finde_spalte(df, cfg.get("hausnr"), ["hausnr", "hausnummer", "hnr"])
        zu = finde_spalte(df, cfg["zusatz"]) if cfg.get("zusatz") else None
        a = None
    else:
        a = finde_spalte(df, cfg.get("adresse"), ["abnahme", "adress"])
        s = h = zu = None

    if z is None or (a is None and (s is None or h is None)):
        print(f"ÜBERSPRUNGEN {name!r}: Zähler={z!r}, Adresse={a!r}, Straße={s!r}, Hausnr={h!r}")
        continue

    if a is not None:
        adr = df[a].fillna("").str.strip()
        print(f"OK {name!r}: Zähler={z!r}, Adresse={a!r}")
    else:
        adr = df[s].fillna("").str.strip() + " " + zahl_sauber(df[h])
        if zu is not None:
            adr = adr + df[zu].fillna("").str.strip()
        print(f"OK {name!r}: Zähler={z!r}, Straße={s!r}, Hausnr={h!r}")

    teile.append(pd.DataFrame({"jahr": jahr, "zaehler": zahl_sauber(df[z]), "adresse_roh": adr.str.strip()}))

if not teile:
    raise ValueError("Kein Blatt konnte gelesen werden. Trage die Spaltennamen in SPALTEN ein.")

lang = pd.concat(teile, ignore_index=True)
lang = lang[(lang["zaehler"] != "") & (lang["adresse_roh"] != "")]

unplausibel = lang[~lang["zaehler"].str.fullmatch(r"\d+")]
print("Zeilen ohne reine Zahl als Zählernummer:", len(unplausibel))
display(unplausibel.sort_values(["jahr", "zaehler"]))
lang = lang[lang["zaehler"].str.fullmatch(r"\d+")]

entfernt = lang[lang["zaehler"].isin(IGNORIEREN)]
print(f"Ignorierte Zähler entfernt: {len(entfernt)} Zeilen von {len(lang)}")
display(entfernt.sort_values(["zaehler", "jahr"])[["zaehler", "jahr", "adresse_roh"]])
lang = lang[~lang["zaehler"].isin(IGNORIEREN)].copy()
lang["adresse_key"] = lang["adresse_roh"].apply(norm_adresse)

pivot = lang.pivot_table(index="adresse_key", columns="jahr", values="zaehler",
                         aggfunc=lambda x: ", ".join(sorted(set(x))))
jahre = sorted(pivot.columns)

def aenderung(row):
    meldungen = []
    for a, b in zip(jahre, jahre[1:]):
        sa = set(row[a].split(", ")) if isinstance(row[a], str) else set()
        sb = set(row[b].split(", ")) if isinstance(row[b], str) else set()
        if not sa and sb:
            meldungen.append(f"{a}: Adresse fehlt")
        elif sa and not sb:
            meldungen.append(f"{b}: Adresse fehlt")
        elif sa != sb:
            meldungen.append(f"{a}→{b}: neu {', '.join(sorted(sb - sa)) or '-'}, entfallen {', '.join(sorted(sa - sb)) or '-'}")
    return " | ".join(meldungen)

ergebnis = pivot.copy()
ergebnis["änderung"] = ergebnis.apply(aenderung, axis=1)
ergebnis.insert(0, "adresse", lang.groupby("adresse_key")["adresse_roh"].first())
ergebnis = ergebnis.reset_index(drop=True)

print("\nAdressen insgesamt:", len(ergebnis))
print("Adressen mit Änderung:", (ergebnis["änderung"] != "").sum())
display(ergebnis.sort_values("änderung", key=lambda c: c == "").head(50))

wandert = lang.groupby("zaehler")["adresse_key"].nunique()
wandert = wandert[wandert > 1]
print("\nZählernummern an mehreren Adressen:", len(wandert))
display(lang[lang["zaehler"].isin(wandert.index)].sort_values(["zaehler", "jahr"])[["zaehler", "jahr", "adresse_roh"]].head(30))

ergebnis.to_csv("zaehler_adressen.csv", **CSV)

## 5 Abgleich mit `Nodes_Edges.ods`

In [ ]:
ne_blaetter = pd.read_excel(NODES_EDGES, sheet_name=None, engine="odf", dtype=str)
for name, df in ne_blaetter.items():
    print(f"Blatt {name!r} ({len(df)} Zeilen): {list(df.columns)}")

### Nodes: Zähler, Anschlusswert und Straße gegen die Jahresblätter

In [ ]:
nodes_roh = ne_blaetter["Nodes"]

c_z = finde_spalte(nodes_roh, stichworte=["zähler", "zaehler"])
c_k = finde_spalte(nodes_roh, stichworte=["kunden"])
c_a = finde_spalte(nodes_roh, stichworte=["anschluss"])
c_s = finde_spalte(nodes_roh, stichworte=["stra"])
print("Erkannt:", dict(zaehler=c_z, kunde=c_k, anschlusswert=c_a, strasse=c_s))
if None in (c_z, c_s):
    raise ValueError("Zähler- oder Straßenspalte nicht erkannt, bitte Namen prüfen.")

nodes = pd.DataFrame({
    "zaehler": zahl_sauber(nodes_roh[c_z]),
    "kunde": nodes_roh[c_k].fillna("").str.strip() if c_k else "",
    "anschlusswert": pd.to_numeric(nodes_roh[c_a].str.replace(",", ".", regex=False), errors="coerce") if c_a else pd.NA,
    "strasse_roh": nodes_roh[c_s].fillna("").str.strip(),
})
nodes = nodes[nodes["zaehler"] != ""]

# Zeilen, die keine echte Zählernummer sind (z.B. wiederholte Überschriften)
komisch = nodes[~nodes["zaehler"].str.fullmatch(r"\d+")]
print("\nZeilen ohne reine Zahl als Zählernummer:", len(komisch))
display(komisch)
nodes = nodes[nodes["zaehler"].str.fullmatch(r"\d+") & ~nodes["zaehler"].isin(IGNORIEREN)].copy()

print("\nZähler in nodes:", nodes["zaehler"].nunique(), "| doppelt vorkommende:", int(nodes["zaehler"].duplicated().sum()))
print("Anschlusswert fehlt oder 0:", int((nodes["anschlusswert"].fillna(0) == 0).sum()))
print(nodes["anschlusswert"].describe().round(1).to_dict())

# Unsere Seite: pro Zähler die zuletzt bekannte Adresse
unsere = (lang.sort_values("jahr").groupby("zaehler")
              .agg(adresse_roh=("adresse_roh", "last"), letztes_jahr=("jahr", "last")).reset_index())

nodes["strasse_key"] = nodes["strasse_roh"].apply(nur_strasse)
nodes["hat_hausnr"] = nodes["strasse_roh"].str.contains(r"\d")
unsere["strasse_key"] = unsere["adresse_roh"].apply(nur_strasse)

z_nodes, z_unsere = set(nodes["zaehler"]), set(unsere["zaehler"])
print(f"\nNur in nodes (nicht in den Jahresblättern): {len(z_nodes - z_unsere)}")
print(f"Nur in den Jahresblättern (nicht in nodes): {len(z_unsere - z_nodes)}")
print(f"In beiden: {len(z_nodes & z_unsere)}")

beide = nodes.merge(unsere, on="zaehler", suffixes=("_nodes", "_unsere"))
beide["strasse_gleich"] = beide["strasse_key_nodes"] == beide["strasse_key_unsere"]
beide["aehnlichkeit"] = [round(difflib.SequenceMatcher(None, a, b).ratio(), 2)
                         for a, b in zip(beide["strasse_key_nodes"], beide["strasse_key_unsere"])]
print("Straße gleich:", int(beide["strasse_gleich"].sum()), "| abweichend:", int((~beide["strasse_gleich"]).sum()))
print("Enthält die nodes-Straße schon eine Hausnummer?", beide["hat_hausnr"].value_counts().to_dict())

print("\nAbweichende Straßen (ähnlichste zuerst, meist Schreibweisen):")
display(beide[~beide["strasse_gleich"]].sort_values("aehnlichkeit", ascending=False)
        [["zaehler", "strasse_roh", "adresse_roh", "aehnlichkeit"]].head(30))
print("\nZähler nur in nodes:")
display(nodes[nodes["zaehler"].isin(z_nodes - z_unsere)][["zaehler", "strasse_roh", "anschlusswert"]].head(30))
print("\nZähler nur in den Jahresblättern:")
display(unsere[unsere["zaehler"].isin(z_unsere - z_nodes)][["zaehler", "adresse_roh", "letztes_jahr"]].head(30))

beide.to_csv("abgleich_nodes_zaehler.csv", **CSV)

### Stammliste: Zähler und HAST-Adressen

In [ ]:
ZUSATZ = ["66761022"]    # Zähler aus nodes, die mit ihrer eigenen Adresse aus nodes dazukommen

# 1) Basis: pro Zähler die zuletzt bekannte Adresse aus den Jahresblättern
anzeige = lang.groupby("adresse_key")["adresse_roh"].first()
zt = (lang.sort_values("jahr").groupby("zaehler")
          .agg(adresse_key=("adresse_key", "last"), letztes_jahr=("jahr", "last")).reset_index())
zt["adresse"] = zt["adresse_key"].map(anzeige)
zt["quelle"] = "Jahresblatt"
zt["hat_hausnr"] = zt["adresse"].str.contains(r"\d")

# 2) Zusätzliche Zähler mit eigener Adresse aus nodes
for z in ZUSATZ:
    if z in set(zt["zaehler"]):
        print(f"HINWEIS: {z} steht schon in den Jahresblättern, wird dort belassen.")
        continue
    n = nodes[nodes["zaehler"] == z]
    if n.empty:
        print(f"WARNUNG: {z} steht nicht im Blatt nodes, übersprungen.")
        continue
    n = n.iloc[0]
    key = norm_adresse(n["strasse_roh"])
    vorhanden = key in anzeige.index
    zt = pd.concat([zt, pd.DataFrame([{
        "zaehler": z, "adresse_key": key, "letztes_jahr": "nodes",
        "adresse": anzeige[key] if vorhanden else n["strasse_roh"],
        "quelle": "nodes", "hat_hausnr": bool(n["hat_hausnr"])}])], ignore_index=True)
    print(f"Kontrolle {z}: Adresse aus nodes = {n['strasse_roh']!r} | "
          f"gibt es diese Adresse schon in den Jahresblättern? {'ja' if vorhanden else 'nein, neue Adresse'} | "
          f"Hausnummer vorhanden? {'ja' if n['hat_hausnr'] else 'NEIN, nur Straße'}")

# 3) Anschlusswert aus nodes anhängen
zt["anschlusswert"] = zt["zaehler"].map(nodes.groupby("zaehler")["anschlusswert"].first())

# 4) Was aus nodes bewusst nicht übernommen wird
nur_nodes = set(nodes["zaehler"]) - set(zt["zaehler"])
print(f"\nZähler nur in nodes, bewusst ignoriert: {len(nur_nodes)}")
display(nodes[nodes["zaehler"].isin(nur_nodes)][["zaehler", "anschlusswert", "strasse_roh"]])

ohne_aw = zt[zt["anschlusswert"].isna()].copy()
ohne_aw["in_nodes"] = ohne_aw["zaehler"].isin(set(nodes["zaehler"]))
print(f"Zähler gesamt: {len(zt)} | ohne Anschlusswert: {len(ohne_aw)}",
      f"(gar nicht in nodes: {int((~ohne_aw['in_nodes']).sum())}, in nodes ohne Wert: {int(ohne_aw['in_nodes'].sum())})")
display(ohne_aw[["zaehler", "adresse", "quelle", "in_nodes"]])
display(zt[zt["quelle"] != "Jahresblatt"])

# 5) Eine Zeile pro Adresse
hast_adressen = (zt.groupby("adresse")
                   .agg(zaehler_aktuell=("zaehler", lambda x: ", ".join(sorted(x))),
                        anzahl_zaehler=("zaehler", "nunique"),
                        anschlusswert_summe=("anschlusswert", lambda x: x.sum(min_count=1)),
                        ohne_anschlusswert=("anschlusswert", lambda x: int(x.isna().sum())),
                        ohne_hausnummer=("hat_hausnr", lambda x: not x.all()),
                        aus_nodes=("quelle", lambda x: bool((x == "nodes").any())))
                   .reset_index())
print("\nAdressen:", len(hast_adressen), "| Mehrfamilienhäuser (mehr als 1 Zähler):", int((hast_adressen["anzahl_zaehler"] > 1).sum()))
print("Anschlusswert gesamt:", round(hast_adressen["anschlusswert_summe"].sum(), 1))
display(hast_adressen.sort_values("anzahl_zaehler", ascending=False).head(15))

zt.to_csv("hast_zaehler.csv", **CSV)
hast_adressen.to_csv("hast_adressen.csv", **CSV)

### Edges: Kanten gegen die Stammliste

In [ ]:
edges_roh = ne_blaetter["Edges"]
c_von, c_nach = finde_spalte(edges_roh, "Node From"), finde_spalte(edges_roh, "Node To")
if c_von is None or c_nach is None:
    raise ValueError(f"Spalten 'Node From' / 'Node To' nicht gefunden. Vorhanden: {list(edges_roh.columns)}")

e = edges_roh[[c_von, c_nach]].rename(columns={c_von: "von_roh", c_nach: "nach_roh"}).dropna(how="all").copy()
e["von_key"], e["nach_key"] = e["von_roh"].apply(adress_key), e["nach_roh"].apply(adress_key)
roh = {**dict(zip(e["von_key"], e["von_roh"])), **dict(zip(e["nach_key"], e["nach_roh"]))}   # Schlüssel -> Originaltext

stamm_keys = {norm_adresse(a): a for a in hast_adressen["adresse"]}
edge_keys = set(e["von_key"].dropna()) | set(e["nach_key"].dropna())

treffer = edge_keys & set(stamm_keys)
ohne_treffer = sorted(edge_keys - set(stamm_keys))
nicht_in_kanten = sorted(set(stamm_keys) - edge_keys)

print(f"Kanten: {len(e)} | verschiedene Nodes in den Kanten: {len(edge_keys)}")
print(f"  davon in der Stammliste: {len(treffer)}")
print(f"  davon NICHT in der Stammliste (Heizwerk, Abzweige, Schächte oder Schreibweise): {len(ohne_treffer)}")
print(f"Adressen der Stammliste, die in keiner Kante vorkommen: {len(nicht_in_kanten)}")

# Nodes ohne Treffer: ähnliche Adresse aus der Stammliste? Gehören sie zu einem ignorierten nodes-Zähler?
vorschlag = [(difflib.get_close_matches(k, list(stamm_keys), n=1, cutoff=0.85) or [None])[0] for k in ohne_treffer]
ign_keys = set(nodes.loc[nodes["zaehler"].isin(nur_nodes), "strasse_roh"].apply(norm_adresse))
print("\nNodes ohne Treffer:")
display(pd.DataFrame({"node_text": [roh.get(k, k) for k in ohne_treffer],
                      "ignorierter_nodes_zaehler": [k in ign_keys for k in ohne_treffer],
                      "aehnliche_adresse": [stamm_keys.get(v) for v in vorschlag]}))

aw_keys = {norm_adresse(a) for a in ohne_aw["adresse"].dropna()}
print("\nAdressen der Stammliste ohne Kante:")
display(pd.DataFrame({"adresse": [stamm_keys[k] for k in nicht_in_kanten],
                      "auch_ohne_anschlusswert": [k in aw_keys for k in nicht_in_kanten]}))

e.to_csv("kanten_roh_bereinigt.csv", **CSV)

### Gerichteter Graph aus den Kanten

In [ ]:
print("Schleifen (von = nach):", int((e["von_key"] == e["nach_key"]).sum()))
print("Doppelte Kanten:", int(e.duplicated(["von_key", "nach_key"]).sum()))

ek = e.dropna(subset=["von_key", "nach_key"])
ek = ek[ek["von_key"] != ek["nach_key"]].drop_duplicates(["von_key", "nach_key"])
G = nx.DiGraph()
G.add_edges_from(zip(ek["von_key"], ek["nach_key"]))

print("Kanten, die es auch in Gegenrichtung gibt:", sum(1 for v, n in G.edges if G.has_edge(n, v) and v < n))
teilnetze = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
print(f"Graph: {G.number_of_nodes()} Nodes, {G.number_of_edges()} Kanten, {len(teilnetze)} Teilnetz(e), "
      f"Teilnetz-Größen: {[len(c) for c in teilnetze][:10]}")
print("Unabhängige Maschen (Ringe) im Netz:", G.number_of_edges() - G.number_of_nodes() + len(teilnetze))
print("Ohne gerichteten Kreis:", nx.is_directed_acyclic_graph(G))

quellen = [n for n in G if G.in_degree(n) == 0]
zusammen = [n for n in G if G.in_degree(n) > 1]
print(f"Quellen (nichts fließt hinein): {len(quellen)} | Nodes mit mehreren Zuflüssen: {len(zusammen)}")
display(pd.DataFrame({"quelle": [roh.get(q, q) for q in quellen],
                      "in_stammliste": [q in stamm_keys for q in quellen],
                      "abgaenge": [G.out_degree(q) for q in quellen]}))
display(pd.DataFrame({"node_mit_mehreren_zufluessen": [roh.get(n, n) for n in zusammen],
                      "zufluesse": [G.in_degree(n) for n in zusammen]}).head(20))

In [ ]:
# Schema: Ebene = längster Weg von einer Quelle
if nx.is_directed_acyclic_graph(G):
    for n in G:
        G.nodes[n]["ebene"] = 0
    for n in nx.topological_sort(G):
        for nachfolger in G.successors(n):
            G.nodes[nachfolger]["ebene"] = max(G.nodes[nachfolger]["ebene"], G.nodes[n]["ebene"] + 1)
    layout = nx.multipartite_layout(G, subset_key="ebene", align="vertical")
else:
    layout = nx.spring_layout(G, seed=1, k=0.6)

farbe = ["tab:red" if n in quellen else ("tab:green" if n in stamm_keys else "lightgray") for n in G]
fig, ax = plt.subplots(figsize=(14, 9))
nx.draw_networkx_edges(G, layout, ax=ax, arrows=True, arrowsize=10, edge_color="gray", node_size=120)
nx.draw_networkx_nodes(G, layout, ax=ax, node_color=farbe, node_size=120, edgecolors="black", linewidths=0.5)
nx.draw_networkx_labels(G, layout, ax=ax, labels={n: roh.get(n, n)[:22] for n in G}, font_size=6)
ax.set_title("Netzschema aus den Kanten (rot = Quelle, grün = HAST aus Stammliste, grau = ohne Adresse in der Stammliste)")
ax.axis("off")
plt.tight_layout()
plt.savefig("netzschema.png", dpi=150)
plt.show()

## 6 Positionen der Adressen

Die Koordinaten stammen aus der Handliste `adressen_koordinaten.ods` und aus nachgetragenen Listen `fehlende_koordinaten_*.ods` (Spalte `koordinate`, z.B. `49.81, 10.34`).
Nominatim fand bei den meisten Adressen nur die Straße, deshalb ist die Suche dort nur noch optional.

In [ ]:
NOMINATIM_SUCHEN = False      # True = Adressen neu bei Nominatim suchen (dauert gut 1 s pro Adresse)
ORT, PLZ = "Wiesentheid", "97353"
VORLAGE = KOORDINATEN

if NOMINATIM_SUCHEN:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter

    geocode = RateLimiter(Nominatim(user_agent="fernwaerme-hast-zuordnung").geocode, min_delay_seconds=1.1)

    def norm_ort(s):
        s = str(s).lower()
        for a, b in UMLAUTE.items():
            s = s.replace(a, b)
        return re.sub(r"[^a-z0-9]+", " ", s).strip()

    def ort_passt(r):
        adr = r.raw.get("address", {})
        gemeinde = [adr.get(k) for k in ("city", "town", "village", "municipality", "suburb", "hamlet")]
        ok_ort = any(g and norm_ort(ORT) in norm_ort(g) for g in gemeinde)
        ok_plz = (not PLZ) or (adr.get("postcode") == PLZ)
        return ok_ort and ok_plz, " / ".join(g for g in gemeinde if g)

    neu = []
    for a in ergebnis["adresse"].dropna().unique():
        strasse, nr = teile_adresse(a)
        anfrage = {"street": f"{nr} {strasse}".strip(), "city": ORT, "country": "Deutschland"}
        if PLZ:
            anfrage["postalcode"] = PLZ
        try:
            r = geocode(anfrage, addressdetails=True)
        except Exception as ex:
            print("Fehler:", type(ex).__name__, ex)
            r = None
        ok, gefundener_ort = ort_passt(r) if r else (False, None)
        neu.append({"adresse": a,
                    "lat": r.latitude if r else None, "lon": r.longitude if r else None,
                    "treffer": r.address if r else None, "ort_gefunden": gefundener_ort, "ort_ok": ok,
                    "genau": ok and bool(r.raw.get("address", {}).get("house_number"))})   # Hausnummer UND richtiger Ort

    nominatim = pd.DataFrame(neu)
    nominatim.to_csv("nominatim_ergebnisse.csv", **CSV)
    print("Genau und im richtigen Ort:", int(nominatim["genau"].sum()))
    print("Treffer im falschen Ort (verworfen):", int((nominatim["lat"].notna() & ~nominatim["ort_ok"]).sum()))
    print("Nur Straße, keine Hausnummer:", int((nominatim["ort_ok"] & ~nominatim["genau"]).sum()))
    print("Nichts gefunden:", int(nominatim["lat"].isna().sum()))

    # Vorlage zum Nachtragen per Hand, nur wenn es noch keine gibt (sonst gingen die Einträge verloren)
    if VORLAGE.exists():
        print(f"{VORLAGE} existiert schon und wird nicht überschrieben.")
    else:
        genau = nominatim[nominatim["genau"]]
        vorlage = (ergebnis[["adresse"]].dropna().drop_duplicates()
                   .merge(genau.assign(koordinate=genau["lat"].round(6).astype(str) + ", " + genau["lon"].round(6).astype(str),
                                       quelle="Nominatim (bitte auf der Karte prüfen)")[["adresse", "koordinate", "quelle"]],
                          on="adresse", how="left")
                   .fillna({"koordinate": "", "quelle": ""})
                   .sort_values("adresse"))
        vorlage.to_excel(VORLAGE, engine="odf", index=False)
        print(len(vorlage), "Adressen nach", VORLAGE, "exportiert")

In [ ]:
RAND_GRAD = 0.05      # so weit dürfen Koordinaten außerhalb des KML-Gebiets liegen

dateien = [VORLAGE]      # vorher: [VORLAGE, *sorted(Path(".").glob("fehlende_koordinaten_*.ods"))]
eingabe = pd.concat([pd.read_excel(p, engine="odf", dtype=str).assign(datei=p.name) for p in dateien if p.exists()],
                    ignore_index=True)
eingabe = eingabe[eingabe["koordinate"].fillna("").str.strip() != ""].copy()
eingabe[["lat", "lon"]] = eingabe["koordinate"].apply(lambda s: pd.Series(parse_koordinate(s), dtype=float))

unlesbar = eingabe[eingabe["lat"].isna()]
print("Gelesen:", {p.name: int((eingabe["datei"] == p.name).sum()) for p in dateien if p.exists()})
print("Nicht lesbar:", len(unlesbar))
display(unlesbar[["datei", "adresse", "koordinate"]])

tausch = (eingabe["lat"] < 20) & (eingabe["lon"] > 40)       # Breite und Länge vertauscht
eingabe.loc[tausch, ["lat", "lon"]] = eingabe.loc[tausch, ["lon", "lat"]].values
print("Automatisch getauscht (Breite/Länge vertauscht):", int(tausch.sum()))

b = kml_linien.to_crs(4326).total_bounds
weit = eingabe[(eingabe["lon"] < b[0] - RAND_GRAD) | (eingabe["lon"] > b[2] + RAND_GRAD) |
               (eingabe["lat"] < b[1] - RAND_GRAD) | (eingabe["lat"] > b[3] + RAND_GRAD)]
print("Weit außerhalb des KML-Gebiets (bitte prüfen):", len(weit))
display(weit[["datei", "adresse", "koordinate"]])

eingabe = eingabe.dropna(subset=["lat", "lon"])
cache = eingabe[["adresse", "lat", "lon"]].drop_duplicates("adresse", keep="last").reset_index(drop=True)
cache["key"] = cache["adresse"].apply(norm_adresse)
cache.to_csv("geocode_cache.csv", **CSV)

# Nachgetragene Listen enthalten die Zähler; diese Koordinaten gelten direkt pro Zähler
koord_zaehler = pd.DataFrame(
    [{"z": norm_z(z), "lat": r.lat, "lon": r.lon}
     for r in eingabe.itertuples() if isinstance(getattr(r, "zaehler", None), str)
     for z in r.zaehler.split(",") if norm_z(z)],
    columns=["z", "lat", "lon"]).drop_duplicates("z", keep="last").set_index("z")

print(f"Adressen mit Position: {len(cache)} | Zähler mit eigener Position: {len(koord_zaehler)}")

### Adressen den Rohrenden zuordnen

Jede Adresse bekommt höchstens ein Rohrende, jedes Rohrende höchstens eine Adresse (nächste Paare zuerst).

In [ ]:
MAX_SNAP_M = 40      # bis zu diesem Abstand wird eine Adresse einem Rohrende zugeordnet
UMKREIS_M  = 200     # Leitungen und freie Enden werden nur in diesem Umkreis um die HASTs gezeigt

# 1) Stammliste: alle Adressen mit Zählern (Zähler aus dem letzten Jahr, in dem die Adresse vorkommt)
stamm = ergebnis[["adresse", "änderung"]].copy()
stamm["zaehler_aktuell"] = ergebnis[jahre[::-1]].bfill(axis=1).iloc[:, 0]
stamm["anzahl_zaehler"] = stamm["zaehler_aktuell"].fillna("").apply(lambda s: len([x for x in s.split(", ") if x]))
stamm = stamm.drop_duplicates("adresse").merge(cache[["adresse", "lat", "lon"]], on="adresse", how="left")
ohne_pos = stamm[stamm["lat"].isna()].copy()
mit_pos = stamm[stamm["lat"].notna()].reset_index(drop=True)
g = punkte_gdf(mit_pos).to_crs(hast.crs)

# 2) Nächste Paare Adresse/Rohrende
enden = hast.reset_index(drop=True).copy()
enden["eid"] = enden.index
A = np.c_[g.geometry.x, g.geometry.y]
E = np.c_[enden.geometry.x, enden.geometry.y]
D = np.hypot(A[:, [0]] - E[:, 0], A[:, [1]] - E[:, 1])
paare = sorted((D[i, j], i, j) for i, j in zip(*np.where(D <= MAX_SNAP_M)))

used_a, used_e, zuord = set(), set(), {}
for d, i, j in paare:
    if i in used_a or j in used_e:
        continue
    used_a.add(i)
    used_e.add(j)
    zuord[i] = (j, d)

# 3) Ergebnisliste aller Adressen mit Position
rows = []
for i, r in g.iterrows():
    zeile = {"adresse": r["adresse"], "anzahl_zaehler": r["anzahl_zaehler"], "zaehler": r["zaehler_aktuell"],
             "änderung": r["änderung"]}
    if i in zuord:
        j, d = zuord[i]
        ende = enden.iloc[j]
        zeile.update(status="am Rohrende", ende_id=int(ende["eid"]), abstand_m=round(d, 1),
                     rohrart=ende["rohrart"], dn=ende["dn"], geometry=ende.geometry)
    else:
        zeile.update(status=f"kein Rohrende in {MAX_SNAP_M} m (Rohr fehlt?)", ende_id=None, abstand_m=None,
                     rohrart=None, dn=None, geometry=r.geometry)
    rows.append(zeile)
hast_alle = gpd.GeoDataFrame(rows, crs=g.crs)

# 4) Rohrenden ohne Adresse (nur im Umfeld der HASTs)
gebiet = hast_alle.geometry.union_all().buffer(UMKREIS_M)
enden_frei = enden[~enden["eid"].isin(used_e) & enden.within(gebiet)].copy()
kml_gebiet = kml[kml.intersects(gebiet)].copy()

print("Adressen insgesamt:", len(stamm))
print("  mit Position:", len(mit_pos), "| ohne Position (fehlen noch in der Tabelle):", len(ohne_pos))
print("  am Rohrende:", int((hast_alle["status"] == "am Rohrende").sum()),
      "| ohne Rohrende in der Nähe:", int((hast_alle["status"] != "am Rohrende").sum()))
print("Rohrenden im Umfeld ohne Adresse:", len(enden_frei))
print("Zähler gesamt (Adressen mit Position):", int(hast_alle["anzahl_zaehler"].sum()))

hast_alle.drop(columns="geometry").assign(rechtswert=hast_alle.geometry.x.round(2),
                                          hochwert=hast_alle.geometry.y.round(2)).to_csv("hast_positionen.csv", **CSV)
ohne_pos[["adresse", "zaehler_aktuell"]].to_csv("hast_ohne_position.csv", **CSV)

# 5) Karte
m = kml_gebiet.to_crs(4326).explore(column="rohrart", categorical=True, name="Leitungen (KML mit Rohrdaten)",
                                    tooltip=["rohrart", "TEXT_LBL", "dn", "mantel", "kml_laenge_m"], style_kwds={"weight": 4})
ns = nur_shp[nur_shp.intersects(gebiet)]
if len(ns):
    ns.to_crs(4326).explore(m=m, color="gray", name="Nur im Shapefile", tooltip=["TEXT_LBL", "laenge_m"],
                            style_kwds={"weight": 2, "dashArray": "6 6"})

fg1 = folium.FeatureGroup(name="HAST (Adresse + Zähler)").add_to(m)
for _, r in hast_alle.to_crs(4326).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=6 + 2 * min(int(r["anzahl_zaehler"]) - 1, 4),
                        color="black", weight=1, fill=True, fill_opacity=0.95,
                        fill_color="green" if r["status"] == "am Rohrende" else "orange",
                        tooltip=f"{r['adresse']} | {r['anzahl_zaehler']} Zähler: {r['zaehler']} | {r['status']}").add_to(fg1)

fg2 = folium.FeatureGroup(name="Rohrenden ohne Adresse").add_to(m)
for _, r in enden_frei.to_crs(4326).iterrows():
    folium.CircleMarker([r.geometry.y, r.geometry.x], radius=4, color="dimgray", weight=1, fill=True,
                        fill_color="white", fill_opacity=0.9,
                        tooltip=f"Ende {r['eid']} | {r['rohrart']} | DN {r['dn']}").add_to(fg2)

karte_abschliessen(m, hast_alle.to_crs(4326).total_bounds, "karte_hast.html")

## 7 Zähler mit Zeitreihen (Menge X)

X = Zähler, für die es eine Zeitreihe in `data/processed/real_meters` gibt und die in der Zuordnung oder in nodes stehen.
Gruppe E: der Zähler hängt an einem Node der Kanten, Gruppe N: ohne Kante.

In [ ]:
MODUS = "zuordnung_oder_nodes"      # oder "alle_drei" (Zeitreihe + Zuordnung + nodes)

# 1) Zähler mit Zeitreihen aus den Dateinamen
dateien = [p for p in REAL_METERS.rglob("*") if p.is_file()]
real = pd.DataFrame({"datei": [p.name for p in dateien], "name": [p.stem for p in dateien]})
real["zaehler"] = real["name"].apply(norm_z)
print("Dateien im Ordner:", len(real), "| verschiedene Zähler:", real["zaehler"].nunique())
print("Dateinamen, die keine reine Zahl sind (bitte prüfen):", int((~real["name"].str.fullmatch(r"\d+")).sum()))
display(real[~real["name"].str.fullmatch(r"\d+")].head(15))
print("Zähler mit mehreren Dateien:", int(real["zaehler"].duplicated().sum()))

# 2) Mengen der Quellen
R = set(real["zaehler"]) - {""}
Z = set(lang["zaehler"].apply(norm_z)) - {""}
N = set(nodes["zaehler"].apply(norm_z)) - {""}
regionen = {
    "nur Zeitreihe (weder Zuordnung noch nodes)": R - Z - N,
    "Zeitreihe + Zuordnung, nicht nodes": (R & Z) - N,
    "Zeitreihe + nodes, nicht Zuordnung": (R & N) - Z,
    "Zeitreihe + Zuordnung + nodes": R & Z & N,
    "Zuordnung oder nodes, ohne Zeitreihe": (Z | N) - R,
}
print("\nÜberblick (Zähler):")
display(pd.DataFrame({"anzahl": {k: len(v) for k, v in regionen.items()}}))

X = (R & Z & N) if MODUS == "alle_drei" else (R & (Z | N))
ign = {norm_z(z) for z in IGNORIEREN}
print("Aus X entfernt (IGNORIEREN):", len(X & ign))
X = X - ign
print(f"\nMenge X ({MODUS}): {len(X)} Zähler")

# 3) Adressen je Zähler und Abgleich mit den Kanten
lang2 = lang.assign(z=lang["zaehler"].apply(norm_z))
keys_z = lang2.groupby("z")["adresse_key"].agg(lambda s: set(s.dropna()))
adr_z = lang2.sort_values("jahr").groupby("z")["adresse_roh"].last()

nodes2 = nodes.assign(z=nodes["zaehler"].apply(norm_z), key=nodes["strasse_roh"].apply(norm_adresse))
keys_n = nodes2.groupby("z")["key"].agg(set)
adr_n = nodes2.groupby("z")["strasse_roh"].first()

zeilen = []
for z in sorted(X):
    kz, kn = keys_z.get(z, set()), keys_n.get(z, set())
    an_kante = (kz | kn) & edge_keys
    zeilen.append({"zaehler": z, "adresse": adr_z.get(z) or adr_n.get(z),
                   "in_zuordnung": z in Z, "in_nodes": z in N,
                   "hat_adresse": bool(kz | kn), "hat_edge": bool(an_kante),
                   "edge_node": next(iter(an_kante), None)})
tab = pd.DataFrame(zeilen)
tab["gruppe"] = tab["hat_edge"].map({True: "E (mit Kante)", False: "N (ohne Kante)"})

x_edge, x_ohne = tab[tab["hat_edge"]], tab[~tab["hat_edge"]]
print(f"E (mit Kante): {len(x_edge)} | N (ohne Kante): {len(x_ohne)}")
print("  davon N ohne jede Adresse:", int((~x_ohne["hat_adresse"]).sum()))
print("  Kanten-Nodes, an denen mehrere X-Zähler hängen (Mehrfamilienhäuser):",
      int((x_edge["edge_node"].value_counts() > 1).sum()))
print("\nN (ohne Kante):")
display(x_ohne[["zaehler", "adresse", "hat_adresse", "in_zuordnung", "in_nodes"]])

Koordinaten an die Zähler hängen. Fehlen welche (auch bei Kanten-Nodes ohne Zähler), wird eine neue Liste `fehlende_koordinaten_<Zeit>.ods` zum Nachtragen geschrieben; nach dem Ausfüllen Abschnitt 6 erneut ausführen.

In [ ]:
# Schlüssel pro Zähler: Node aus den Kanten, sonst die bereinigte Adresse
tab["key"] = [n if isinstance(n, str) else adress_key(a) for n, a in zip(tab["edge_node"], tab["adresse"])]
tab = tab.merge(cache.drop_duplicates("key")[["key", "lat", "lon"]], on="key", how="left")
for sp in ("lat", "lon"):
    tab[sp] = tab[sp].fillna(tab["zaehler"].map(koord_zaehler[sp]))
tab["hat_koord"] = tab["lat"].notna()

print("Zähler in X:", len(tab), "| mit Koordinate:", int(tab["hat_koord"].sum()))
print(tab.groupby("gruppe")["hat_koord"].agg(gesamt="size", mit_koordinate="sum"))
print("Verschiedene Adressen in E:", tab.loc[tab["hat_edge"], "key"].nunique(),
      "| davon mit Koordinate:", tab.loc[tab["hat_edge"] & tab["hat_koord"], "key"].nunique())

# Fehlende Positionen: Zähler aus X und Kanten-Nodes ohne Zähler (z.B. Häuser, über die die Leitung weiterläuft)
fehlt = tab[~tab["hat_koord"]]
bekannt = set(cache["key"]) | set(tab.loc[tab["hat_koord"], "key"])
nodes_ohne_pos = sorted(n for n in G if n not in bekannt and n not in set(fehlt["key"]))
print("Kanten-Nodes ohne Position:", len(nodes_ohne_pos))

if len(fehlt) or nodes_ohne_pos:
    liste = pd.concat([
        fehlt.groupby("key").agg(adresse=("adresse", "first"), zaehler=("zaehler", ", ".join), gruppe=("gruppe", "first")),
        pd.DataFrame({"adresse": [roh.get(n, n) for n in nodes_ohne_pos], "zaehler": "",
                      "gruppe": "Kanten-Node ohne Zähler"}, index=nodes_ohne_pos),
    ]).reset_index(drop=True).sort_values(["gruppe", "adresse"]).assign(koordinate="")
    dateiname = f"fehlende_koordinaten_{datetime.now():%Y%m%d_%H%M}.ods"
    liste.to_excel(dateiname, engine="odf", index=False)
    print("Geschrieben:", dateiname, "|", len(liste), "Adressen ohne Koordinate")
    display(liste.head(20))

tab.to_csv("x_zaehler_mit_gruppe.csv", **CSV)

In [ ]:
x_edge = tab[tab["hat_edge"]].copy()
print(f"E (mit Kante): {len(x_edge)} Zähler | mit Koordinate: {int(x_edge['hat_koord'].sum())} | ohne: {int((~x_edge['hat_koord']).sum())}")
display(x_edge[~x_edge["hat_koord"]][["zaehler", "adresse", "edge_node"]])

knoten = (x_edge.dropna(subset=["lat", "lon"]).groupby("edge_node")
                .agg(lat=("lat", "mean"), lon=("lon", "mean"), lat_n=("lat", "nunique"),
                     zaehler=("zaehler", ", ".join), anzahl=("zaehler", "nunique"),
                     adresse=("adresse", "first")).reset_index())
print("Nodes mit Zählern aus E:", len(knoten), "| Nodes mit abweichenden Koordinaten pro Zähler:", int((knoten["lat_n"] > 1).sum()))

# Positionen aller Nodes: E-Nodes aus den Zählern, die übrigen aus der Adressliste
cpos = cache.drop_duplicates("key").set_index("key")
koord = {r.edge_node: (r.lat, r.lon) for r in knoten.itertuples()}
for n in G:
    if n not in koord and n in cpos.index:
        koord[n] = (cpos.loc[n, "lat"], cpos.loc[n, "lon"])
e_nodes = set(knoten["edge_node"])

zeichnen = [(v, n) for v, n in G.edges if v in koord and n in koord]
print(f"\nKanten gesamt: {G.number_of_edges()} | mit Position an beiden Enden: {len(zeichnen)} | "
      f"berühren einen E-Node: {sum(1 for v, n in G.edges if v in e_nodes or n in e_nodes)}")
ohne_koord = sorted(set(G) - set(koord))
print("Nodes in den Kanten ohne Position:", len(ohne_koord))
display(pd.DataFrame({"node": [roh.get(n, n) for n in ohne_koord]}))

# Karte
m = folium.Map(location=[knoten["lat"].mean(), knoten["lon"].mean()], zoom_start=17, tiles="OpenStreetMap")
fk = folium.FeatureGroup(name="Kanten (von → nach, Luftlinie)").add_to(m)
for v, n in zeichnen:
    linie = folium.PolyLine([koord[v], koord[n]], color="#d6402d", weight=3, opacity=0.8,
                            tooltip=f"{roh.get(v, v)} → {roh.get(n, n)}").add_to(fk)
    PolyLineTextPath(linie, "     ►     ", repeat=True, offset=7,
                     attributes={"fill": "#d6402d", "font-size": "14"}).add_to(fk)

fn = folium.FeatureGroup(name=f"Zähler aus E ({len(x_edge)})").add_to(m)
for r in knoten.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=6 + 2 * min(r.anzahl - 1, 4), color="black", weight=1,
                        fill=True, fill_color="green", fill_opacity=0.95,
                        tooltip=f"{r.adresse} | {r.anzahl} Zähler: {r.zaehler}").add_to(fn)

fo = folium.FeatureGroup(name="Andere Nodes mit Position").add_to(m)
for n, (la, lo) in koord.items():
    if n not in e_nodes:
        folium.CircleMarker([la, lo], radius=4, color="dimgray", weight=1, fill=True, fill_color="lightgray",
                            fill_opacity=0.9, tooltip=roh.get(n, n)).add_to(fo)

la, lo = zip(*koord.values())
karte_abschliessen(m, [min(lo), min(la), max(lo), max(la)], "karte_kanten_E.html")
display(m)

# Export der E-Zähler mit UTM-Koordinaten (passend zu Shapefile und KML)
x_pos = punkte_gdf(x_edge.dropna(subset=["lat", "lon"])).to_crs(CRS_UTM)
x_pos.drop(columns="geometry").assign(rechtswert=x_pos.geometry.x.round(2),
                                      hochwert=x_pos.geometry.y.round(2)).to_csv("x83_mit_koordinaten.csv", **CSV)

## 8 Kanten über die Rohre legen

Statt der Luftlinie wird jede Kante von → nach auf dem kürzesten Weg durch das KML-Leitungsnetz gezeichnet.

- Die Leitungen werden an allen Kreuzungen geteilt. Offene Leitungsenden rasten auf das nächste Rohr eines anderen Teilnetzes ein:
  bis `SNAP_M` gelten sie als verbunden, bis `MAX_LUECKE_M` wird die Lücke überbrückt, zählt beim Routing aber deutlich teurer und wird ausgewiesen.
- Start- und Zielpunkt einer Kante: das zugeordnete Rohrende der HAST (Abschnitt 6), sonst die Adresskoordinate, eingerastet auf das nächste Rohr.

In [ ]:
SNAP_M          = 1.0    # Lücken bis zu dieser Länge gelten als verbunden (Zeichenungenauigkeit)
MAX_LUECKE_M    = 25     # größere Lücken werden bis zu dieser Länge überbrückt, aber als Lücke markiert
LUECKE_FAKTOR   = 5      # eine überbrückte Lücke zählt beim Routing so viel mal teurer als ein echtes Rohr
MAX_ANBINDUNG_M = 50     # so weit darf eine Node-Position höchstens vom Netz entfernt sein

def pkt(c):
    return (round(c[0], 3), round(c[1], 3))

# 1) Leitungen an allen Kreuzungen teilen und Teilnetze bestimmen
segs = [s for s in unary_union(list(kml.geometry[kml.geom_type == "LineString"])).geoms if s.length > 0]
seg_baum = shapely.STRtree(segs)

roh_netz = nx.Graph((pkt(s.coords[0]), pkt(s.coords[-1])) for s in segs)
komp = {n: k for k, c in enumerate(nx.connected_components(roh_netz)) for n in c}
seg_komp = [komp[pkt(s.coords[0])] for s in segs]

# 2) Verbinder sammeln: (Punkt, Segment, Stelle entlang des Segments, Typ, Länge)
verbinder = []
for p in (n for n in roh_netz if roh_netz.degree(n) == 1):          # offene Enden
    pt = Point(p)
    kand = [(segs[j].distance(pt), j) for j in seg_baum.query(pt, predicate="dwithin", distance=MAX_LUECKE_M)
            if seg_komp[j] != komp[p]]
    if kand:
        d, j = min(kand)
        verbinder.append((p, j, segs[j].project(pt), "Verbindung" if d <= SNAP_M else "Lücke", d))

hast_rohrende = {norm_adresse(r.adresse): r.geometry for r in hast_alle.itertuples() if r.status == "am Rohrende"}
koord_utm = gpd.GeoSeries([Point(lo, la) for la, lo in koord.values()], index=list(koord), crs=4326).to_crs(CRS_UTM)

anker = {}                                                           # Kanten-Node -> Punkt im Netz
for n in G:
    q = hast_rohrende.get(n, koord_utm.get(n))
    if q is None:
        continue
    j = seg_baum.query_nearest(q)[0]
    if segs[j].distance(q) <= MAX_ANBINDUNG_M:
        anker[n] = pkt(q.coords[0])
        verbinder.append((anker[n], j, segs[j].project(q), "Anbindung", segs[j].distance(q)))

# 3) Routing-Graph: Rohrstücke (an den Einrastpunkten geteilt) plus Verbinder
teilungen = {}
for _, j, stelle, _, _ in verbinder:
    teilungen.setdefault(j, set()).add(stelle)

netz = nx.Graph()
for i, s in enumerate(segs):
    ds = sorted(d for d in teilungen.get(i, ()) if 0 < d < s.length)
    stellen = [0.0, *ds, s.length]
    punkte = [pkt(s.coords[0]), *[pkt(s.interpolate(d).coords[0]) for d in ds], pkt(s.coords[-1])]
    for a, b, pa, pb in zip(stellen, stellen[1:], punkte, punkte[1:]):
        if pa != pb:
            netz.add_edge(pa, pb, w=b - a, typ="Rohr", geom=substring(s, a, b))
for p, j, stelle, typ, d in verbinder:
    ziel = pkt(segs[j].interpolate(stelle).coords[0])
    if p != ziel:
        netz.add_edge(p, ziel, w=d * (LUECKE_FAKTOR if typ == "Lücke" else 1), typ=typ, geom=LineString([p, ziel]))

# 4) Jede Kante auf dem kürzesten Weg durch das Netz
def route(a, b):
    pfad = nx.shortest_path(netz, a, b, weight="w")
    coords, luecke = [], 0.0
    for u, v in zip(pfad, pfad[1:]):
        dat = netz.edges[u, v]
        c = list(dat["geom"].coords)
        if pkt(c[0]) != u:
            c = c[::-1]
        coords += c if not coords else c[1:]
        if dat["typ"] == "Lücke":
            luecke += dat["geom"].length
    return LineString(coords), luecke

zeilen = []
for v, n in G.edges:
    zeile = {"von": roh.get(v, v), "nach": roh.get(n, n)}
    if v in anker and n in anker:
        luft = Point(anker[v]).distance(Point(anker[n]))
        try:
            geom, luecke = route(anker[v], anker[n])
            zeile.update(status="über Rohre" if luecke == 0 else "über Rohre mit Lücke", laenge_rohr_m=round(geom.length, 1),
                         luftlinie_m=round(luft, 1), luecke_m=round(luecke, 1), geometry=geom)
        except nx.NetworkXNoPath:
            zeile.update(status="kein Weg im Netz (Luftlinie)", luftlinie_m=round(luft, 1),
                         geometry=LineString([anker[v], anker[n]]))
    else:
        zeile.update(status="Node ohne Position", geometry=None)
    zeilen.append(zeile)
kanten_rohr = gpd.GeoDataFrame(zeilen, geometry="geometry", crs=CRS_UTM)
kanten_rohr["umweg"] = (kanten_rohr["laenge_rohr_m"] / kanten_rohr["luftlinie_m"]).round(1)

print("Rohrstücke:", len(segs), "| Teilnetze vor dem Einrasten:", len(set(komp.values())))
print("Offene Enden eingerastet:", sum(t != "Anbindung" for *_, t, _ in verbinder),
      "| davon als Lücke überbrückt:", sum(t == "Lücke" for *_, t, _ in verbinder))
print("Kanten-Nodes im Netz verankert:", len(anker), "von", G.number_of_nodes())
print(kanten_rohr["status"].value_counts().to_dict())
print("Umweg (Rohrlänge / Luftlinie):", kanten_rohr["umweg"].describe().round(1).to_dict())

print("\nGrößte Umwege (bitte auf der Karte prüfen):")
display(kanten_rohr.drop(columns="geometry").sort_values("umweg", ascending=False).head(10))
print("Kanten mit überbrückter Lücke oder ohne Weg:")
display(kanten_rohr[~kanten_rohr["status"].isin(["über Rohre", "Node ohne Position"])].drop(columns="geometry"))

kanten_rohr.drop(columns="geometry").to_csv("kanten_auf_rohren.csv", **CSV)
kanten_rohr.dropna(subset=["geometry"]).to_crs(4326).to_file("kanten_auf_rohren.geojson", driver="GeoJSON")

In [ ]:
FARBE = {"über Rohre": "#d6402d", "über Rohre mit Lücke": "#e68a00", "kein Weg im Netz (Luftlinie)": "gray"}

gezeichnet = kanten_rohr.dropna(subset=["geometry"]).to_crs(4326)
gebiet_k = kanten_rohr.dropna(subset=["geometry"]).union_all().buffer(UMKREIS_M)

m = folium.Map(tiles="OpenStreetMap")
kml[kml.intersects(gebiet_k)].to_crs(4326).explore(
    m=m, color="steelblue", name="Leitungen (KML)", tooltip=["rohrart", "TEXT_LBL", "dn"],
    style_kwds={"weight": 2, "opacity": 0.5})

fk = folium.FeatureGroup(name="Kanten über die Rohre (von → nach)").add_to(m)
for r in gezeichnet.itertuples():
    farbe = FARBE[r.status]
    linie = folium.PolyLine([(y, x) for x, y in r.geometry.coords], color=farbe, weight=3, opacity=0.85,
                            dash_array="6 6" if r.status.startswith("kein") else None,
                            tooltip=f"{r.von} → {r.nach} | {r.status} | {r.laenge_rohr_m} m Rohr, "
                                    f"{r.luftlinie_m} m Luftlinie, Lücke {r.luecke_m} m").add_to(fk)
    PolyLineTextPath(linie, "     ►     ", repeat=True, offset=7, attributes={"fill": farbe, "font-size": "14"}).add_to(fk)

fn = folium.FeatureGroup(name="Kanten-Nodes").add_to(m)
anker_wgs = gpd.GeoSeries([Point(p) for p in anker.values()], index=list(anker), crs=CRS_UTM).to_crs(4326)
for n, p in anker_wgs.items():
    folium.CircleMarker([p.y, p.x], radius=5, color="black", weight=1, fill=True, fill_opacity=0.95,
                        fill_color="red" if n in quellen else ("green" if n in e_nodes else "lightgray"),
                        tooltip=roh.get(n, n)).add_to(fn)

karte_abschliessen(m, gezeichnet.total_bounds, "karte_kanten_rohre.html")

In [ ]:


ORDNER = str(REAL_METERS)

FLUSS_MIN_LH    = 10      # Stunden mit Durchfluss bis zu diesem Wert (l/h) werden ausgeblendet. Startwert, bitte prüfen
GEWICHTET       = True    # Tageswert nach Durchfluss gewichtet (sonst Median)
MIN_STUNDEN_TAG = 6       # so viele gültige Stunden braucht ein Tag für einen Tageswert
T_MIN, T_MAX    = 0, 130  # Temperaturen außerhalb gelten als Messfehler
OVERRIDE        = {}      # nur falls eine Spalte nicht erkannt wird, z.B. {"vor": "Flow temperature (°C)"}

MUSTER = {
    "zeit":     r"^\s*(timestamp|zeit|time|datum|date)",
    "vor":      r"^\s*flow\s*temp",
    "rueck":    r"^\s*return\s*temp",
    "diff":     r"temperature\s*diff",
    "fluss":    r"volume\s*flow",
    "leistung": r"^\s*power",
    "energie":  r"^\s*energy",
}

if "norm_z" not in globals():
    def norm_z(s): return re.sub(r"\D", "", str(s)).lstrip("0")

zaehler_liste = sorted(set(x_edge["zaehler"].apply(norm_z)))
pfade = {norm_z(p.stem): p for p in Path(ORDNER).rglob("*.csv")}
ohne_csv = [z for z in zaehler_liste if z not in pfade]
print(f"Zähler in E: {len(zaehler_liste)} | CSV gefunden: {len(zaehler_liste) - len(ohne_csv)} | ohne CSV: {len(ohne_csv)}")
if ohne_csv:
    print("Ohne CSV:", ohne_csv)

def num(s):
    return pd.to_numeric(s.astype(str).str.strip().str.replace(",", ".", regex=False), errors="coerce")

def lies_csv(pfad):
    roh = Path(pfad).read_bytes()
    text = None
    for enc in ("utf-8-sig", "cp1252"):
        try:
            text = roh.decode(enc); break
        except UnicodeDecodeError:
            pass
    if text is None:
        text = roh.decode("latin-1")
    kopf_z = text.splitlines()[:15]
    sep = max([";", ",", "\t", "|"], key=lambda s: sum(z.count(s) for z in kopf_z))
    maxn = max(z.count(sep) for z in kopf_z)
    kopf = next(i for i, z in enumerate(kopf_z) if z.count(sep) == maxn)
    return pd.read_csv(StringIO(text), sep=sep, skiprows=kopf, dtype=str), sep

def zeit_parse(s):
    s = s.astype(str).str.strip()
    if s.str.fullmatch(r"\d{10}|\d{13}").mean() > 0.9:
        t = pd.to_datetime(pd.to_numeric(s, errors="coerce"), unit="ms" if s.str.len().iloc[0] == 13 else "s", errors="coerce")
    elif s.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$").mean() > 0.5:      # mit Zeitzone
        t = pd.to_datetime(s, errors="coerce", utc=True).dt.tz_convert("Europe/Berlin").dt.tz_localize(None)
    else:
        t = pd.to_datetime(s, dayfirst=True, errors="coerce", format="mixed")
    return t

def spalte(cols, muster):
    for c in cols:
        if re.search(muster, str(c), re.I):
            return c
    return None

serien, tage, info = {}, {}, []
for z in zaehler_liste:
    if z not in pfade:
        continue
    try:
        df, sep = lies_csv(pfade[z])
        cols = list(df.columns)
        sp = {k: OVERRIDE.get(k) or spalte(cols, m) for k, m in MUSTER.items()}
        if None in (sp["zeit"], sp["vor"], sp["rueck"]) or sp["vor"] == sp["rueck"]:
            info.append({"zaehler": z, "status": "Spalten nicht erkannt", "spalten": cols})
            continue

        d = pd.DataFrame({"vor": num(df[sp["vor"]]).values, "rueck": num(df[sp["rueck"]]).values},
                         index=pd.DatetimeIndex(zeit_parse(df[sp["zeit"]])))
        for k in ("fluss", "leistung", "diff", "energie"):
            d[k] = num(df[sp[k]]).values if sp[k] else np.nan
        d = d[d.index.notna()].sort_index()
        d = d[~d.index.duplicated(keep="first")]
        d = d.resample("1h").mean()                       # lückenloses Stundenraster, fehlende Stunden = NaN

        # Messfehler entfernen, Spreizung berechnen, Stunden ohne Durchfluss ausblenden
        schlecht_t = (d["vor"].lt(T_MIN) | d["vor"].gt(T_MAX) | d["rueck"].lt(T_MIN) | d["rueck"].gt(T_MAX))
        d.loc[schlecht_t, ["vor", "rueck"]] = np.nan
        d["spreizung"] = d["vor"] - d["rueck"]
        ohne_fluss = d["fluss"].le(FLUSS_MIN_LH) if sp["fluss"] else pd.Series(False, index=d.index)
        d.loc[ohne_fluss, "spreizung"] = np.nan

        # Gegenproben
        luecke = d["vor"].isna() | d["rueck"].isna()
        runs = luecke.ne(luecke.shift()).cumsum()
        laengste = int(luecke.groupby(runs).sum().max())
        abw_geraet = (d["vor"] - d["rueck"] - d["diff"]).abs().median() if sp["diff"] else np.nan
        ok = d["spreizung"].gt(1) & d["fluss"].gt(FLUSS_MIN_LH)
        pv = (d.loc[ok, "leistung"] / (d.loc[ok, "fluss"] * d.loc[ok, "spreizung"] * 0.001163)).median() \
             if (sp["leistung"] and sp["fluss"] and ok.any()) else np.nan

        # Tageswert
        gueltig = d["spreizung"].notna()
        median_t = d["spreizung"].resample("D").median()
        if GEWICHTET and sp["fluss"]:
            w = d["fluss"].where(gueltig)
            t_ = (d["spreizung"] * w).resample("D").sum(min_count=1) / w.resample("D").sum(min_count=1)
            t_ = t_.fillna(median_t)
        else:
            t_ = median_t
        tage[z] = t_.where(gueltig.resample("D").sum() >= MIN_STUNDEN_TAG)
        serien[z] = d

        sp_ok = d["spreizung"].dropna()
        info.append({"zaehler": z, "status": "ok", "von": d.index.min().date(), "bis": d.index.max().date(),
                     "stunden": len(d), "anteil_vorhanden": round(1 - luecke.mean(), 2),
                     "laengste_luecke_h": laengste,
                     "anteil_ohne_fluss": round(ohne_fluss.mean(), 2),
                     "anteil_negativ": round((sp_ok < 0).mean(), 3) if len(sp_ok) else np.nan,
                     "vor_median": round(d["vor"].median(), 1), "rueck_median": round(d["rueck"].median(), 1),
                     "abw_zu_geraet_K": round(abw_geraet, 2) if pd.notna(abw_geraet) else np.nan,
                     "leistung_passt": round(pv, 2) if pd.notna(pv) else np.nan,
                     "trenner": repr(sep),
                     "spalten": " | ".join(str(sp[k]) for k in ("zeit", "vor", "rueck", "fluss", "leistung"))})
    except Exception as ex:
        info.append({"zaehler": z, "status": f"Fehler: {type(ex).__name__}: {ex}"})

ueberblick = pd.DataFrame(info)
tag = pd.DataFrame(tage)

print("\nStatus:", ueberblick["status"].value_counts().to_dict())
if "spalten" in ueberblick:
    print("\nErkannte Spalten (zeit | vor | rueck | fluss | leistung):")
    print(ueberblick["spalten"].astype(str).value_counts().head(5).to_string())
display(ueberblick[ueberblick["status"] != "ok"])

ok_t = ueberblick[ueberblick["status"] == "ok"]
if len(ok_t):
    print(f"\nZeitraum: {ok_t['von'].min()} bis {ok_t['bis'].max()} | Tageswerte: {tag.shape[0]} Tage x {tag.shape[1]} Zähler")
    print("Anteil vorhandener Stunden (Median über alle Zähler):", round(ok_t["anteil_vorhanden"].median(), 2))
    print("Anteil ohne Durchfluss (Median):", round(ok_t["anteil_ohne_fluss"].median(), 2))
    print("Zähler mit weniger als 80 % vorhandenen Stunden:", int((ok_t["anteil_vorhanden"] < 0.8).sum()))
    print("Zähler mit mehr als 10 % negativer Spreizung:", int((ok_t["anteil_negativ"] > 0.1).sum()))
    print("Zähler, bei denen die eigene Spreizung um mehr als 0,5 K vom Zählerwert abweicht:", int((ok_t["abw_zu_geraet_K"] > 0.5).sum()))
    print("Zähler, bei denen Leistung nicht zu Durchfluss und Spreizung passt (Verhältnis außerhalb 0,8 bis 1,2):",
          int(((ok_t["leistung_passt"] < 0.8) | (ok_t["leistung_passt"] > 1.2)).sum()))
    display(ok_t.drop(columns=["spalten", "trenner"]).head(12))
    auffaellig = ok_t[(ok_t["anteil_negativ"] > 0.1) | (ok_t["abw_zu_geraet_K"] > 0.5)
                      | (ok_t["leistung_passt"] < 0.8) | (ok_t["leistung_passt"] > 1.2)]
    print("\nAuffällige Zähler:")
    display(auffaellig[["zaehler", "anteil_negativ", "abw_zu_geraet_K", "leistung_passt", "vor_median", "rueck_median"]])

In [ ]:


adr = tab.drop_duplicates("zaehler").set_index("zaehler")["adresse"].to_dict()

# 1) Der Zähler mit negativer Spreizung
neg = ok_t[ok_t["anteil_negativ"] > 0.1][["zaehler", "anteil_negativ", "vor_median", "rueck_median"]].copy()
neg["adresse"] = neg["zaehler"].map(adr)
display(neg)
print("Zähler ohne Wert bei 'leistung_passt' oder 'abw_zu_geraet_K' (dort lief die Prüfung leer):",
      int(ok_t["leistung_passt"].isna().sum()), "/", int(ok_t["abw_zu_geraet_K"].isna().sum()))

# 2) Wodurch geht jede Stunde verloren?
def kategorien(d):
    keine = d[["vor", "rueck", "fluss", "leistung"]].isna().all(axis=1)          # gar keine Messzeile
    tfehlt = (~keine) & (d["vor"].isna() | d["rueck"].isna())                    # Zeile da, Temperatur fehlt
    kfluss = (~keine) & (~tfehlt) & d["fluss"].le(FLUSS_MIN_LH)                  # Temperatur da, kein Durchfluss
    nutz = d["spreizung"].notna()
    return keine, tfehlt, kfluss, nutz

def luecken(maske):
    runs = maske.ne(maske.shift()).cumsum()
    g = maske.groupby(runs).agg(["first", "sum"])
    return g.loc[g["first"], "sum"]

zeilen, alle_luecken, monat_keine, monat_nutz = [], [], {}, {}
std = pd.DataFrame(0.0, index=range(24), columns=["fehlt", "n"])
for z, d in serien.items():
    keine, tfehlt, kfluss, nutz = kategorien(d)
    zeilen.append({"zaehler": z, "keine_zeile": keine.mean(), "temp_fehlt": tfehlt.mean(),
                   "kein_fluss": kfluss.mean(), "nutzbar": nutz.mean()})
    alle_luecken += list(luecken(keine))
    monat_keine[z] = keine.resample("MS").mean()
    monat_nutz[z] = nutz.resample("MS").mean()
    g = keine.groupby(keine.index.hour).agg(["sum", "count"])
    std["fehlt"] += g["sum"].reindex(range(24)).fillna(0)
    std["n"] += g["count"].reindex(range(24)).fillna(0)

K = pd.DataFrame(zeilen).set_index("zaehler")
print("\nAnteil aller Stunden (Median über die 83 Zähler), in Prozent:")
print((K.median() * 100).round(1).rename({"keine_zeile": "keine Messzeile", "temp_fehlt": "Zeile da, Temperatur fehlt",
                                          "kein_fluss": f"kein Durchfluss (≤ {FLUSS_MIN_LH} l/h)",
                                          "nutzbar": "nutzbar"}).to_string())

# 3) Wie lang sind die Lücken?
L = pd.Series(alle_luecken, dtype=float)
if len(L):
    kl = pd.cut(L, [0, 3, 24, 168, np.inf], labels=["≤ 3 h", "4–24 h", "1–7 Tage", "> 7 Tage"])
    tab_l = pd.DataFrame({"anzahl": L.groupby(kl, observed=False).size(),
                          "anteil_der_fehlenden_stunden_%": (L.groupby(kl, observed=False).sum() / L.sum() * 100).round(0)})
    print("\nLücken ohne Messzeile nach Länge:")
    display(tab_l)

# 4) Durchfluss-Verteilung (zur Kontrolle der Grenze)
f = pd.concat([d["fluss"].dropna() for d in serien.values()])
print("\nDurchfluss in l/h, Quantile 10/25/50/75/90 %:", f.quantile([.1, .25, .5, .75, .9]).round(0).tolist())
print(f"Anteil der vorhandenen Stunden: ≤ 10 l/h: {(f <= 10).mean():.0%} | ≤ 50 l/h: {(f <= 50).mean():.0%}")

# 5) Grafiken: Abdeckung nach Monat und nach Tageszeit
MK, MN = pd.DataFrame(monat_keine), pd.DataFrame(monat_nutz)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]})
ax[0].plot(MK.index, (1 - MK.mean(axis=1)) * 100, label="Stunden mit Messzeile")
ax[0].plot(MN.index, MN.mean(axis=1) * 100, label="nutzbare Stunden")
ax[0].set_ylim(0, 100); ax[0].set_ylabel("% der Stunden (Mittel über Zähler)"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title("Abdeckung nach Monat")
ax[1].bar(std.index, (1 - std["fehlt"] / std["n"]) * 100, color="#2e6db3")
ax[1].set_ylim(0, 100); ax[1].set_xlabel("Stunde des Tages"); ax[1].set_title("Messzeile vorhanden, nach Tageszeit")
plt.tight_layout(); plt.savefig("abdeckung.png", dpi=150); plt.show()

In [ ]:


# 1) Abdeckung pro Zähler und Monat: gemeinsamer Ausfall oder einzelne Zähler?
cov = pd.DataFrame({z: d["vor"].notna().resample("MS").mean() for z, d in serien.items()}).loc["2022-06":]
reihenfolge = cov.loc["2023-08":"2025-05"].mean().sort_values().index
x = mdates.date2num(cov.index.to_pydatetime())
fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(cov[reihenfolge].T.values * 100, aspect="auto", cmap="viridis", vmin=0, vmax=100,
               extent=[x[0], x[-1], len(reihenfolge), 0], interpolation="nearest")
ax.xaxis_date(); ax.set_yticks([]); ax.set_ylabel("83 Zähler")
fig.colorbar(im, ax=ax, label="% der Stunden mit Messzeile")
ax.set_title("Abdeckung pro Zähler und Monat")
plt.tight_layout(); plt.savefig("abdeckung_pro_zaehler.png", dpi=150); plt.show()

# 2) Wann beginnt und endet jeder Zähler, und was kam in der frühen Phase an?
erste = pd.Series({z: d["vor"].first_valid_index() for z, d in serien.items()})
letzte = pd.Series({z: d["vor"].last_valid_index() for z, d in serien.items()})
print("Erste Messzeile je Zähler, nach Monat:")
print(erste.dt.to_period("M").value_counts().sort_index().to_string())
print("\nLetzte Messzeile je Zähler, nach Monat:")
print(letzte.dt.to_period("M").value_counts().sort_index().to_string())

frueh = pd.concat([d.loc[:"2022-09-30"].dropna(subset=["vor"]) for d in serien.values()])
print("\nMesszeilen bis 30.09.2022 insgesamt:", len(frueh))
print("Davon nach Stunde des Tages (häufigste 5):", frueh.index.hour.value_counts().head(5).to_dict())

# 3) Der Zähler mit negativer Spreizung
z = "80912182"
d = serien[z]
s = d["spreizung"].dropna()
print(f"\nZähler {z}: Vorlauf-Median {d['vor'].median():.1f} °C | Rücklauf-Median {d['rueck'].median():.1f} °C")
print("Anteil negativer Spreizung je Quartal:")
print((s < 0).groupby(s.index.to_period("Q")).mean().round(2).to_string())

# 4) Ab welchem Durchfluss ist die Spreizung aussagekräftig?
alle = pd.concat([pd.DataFrame({"s": d["vor"] - d["rueck"], "f": d["fluss"]}).dropna() for d in serien.values()])
klassen = pd.cut(alle["f"], [-0.1, 0.5, 10, 30, 50, 100, 200, 1e9],
                 labels=["0", "bis 10", "10–30", "30–50", "50–100", "100–200", "über 200"])
print("\nSpreizung in K nach Durchfluss (l/h), alle Zähler:")
print(alle.groupby(klassen, observed=False)["s"]
      .describe(percentiles=[.25, .5, .75]).round(1)[["count", "25%", "50%", "75%"]]
      .rename(columns={"25%": "p25", "50%": "median", "75%": "p75"}).to_string())

36 Zähler liefern seit Januar 2021 (zunächst nur den 23-Uhr-Wert), 40 Zähler beginnen im Januar 2023. Dazu kommen 7 Nachzügler. 79 Zähler laufen bis Dezember 2025, 4 enden früher (je 2 im Dezember 2024 und im März 2025)

In [ ]:
FLUSS_MIN_LH    = 0.5
MIN_STUNDEN_TAG = 6
GEWICHTET       = True

# 1) Spreizung und Tageswerte aus den geladenen Reihen neu berechnen
tage = {}
for z, d in serien.items():
    s = (d["vor"] - d["rueck"]).where(d["fluss"] > FLUSS_MIN_LH)
    d["spreizung"] = s
    g = s.notna()
    med = s.resample("D").median()
    if GEWICHTET:
        w = d["fluss"].where(g)
        t_ = (s * w).resample("D").sum(min_count=1) / w.resample("D").sum(min_count=1)
        t_ = t_.fillna(med)
    else:
        t_ = med
    tage[z] = t_.where(g.resample("D").sum() >= MIN_STUNDEN_TAG)
tag = pd.DataFrame(tage)
print("Tageswerte:", tag.shape[0], "Tage x", tag.shape[1], "Zähler")

# 2) Phasen: fehlen einzelne Stunden oder ganze Tage?
phasen = {"Jan 2024 – Nov 2024": ("2024-01-01", "2024-11-30"),
          "Dez 2024 – Mai 2025": ("2024-12-01", "2025-05-31"),
          "Jun 2025 – Nov 2025": ("2025-06-01", "2025-11-30")}
zeilen = []
for name, (a, b) in phasen.items():
    h_alle, h_mit, anteil = [], [], []
    for z, d in serien.items():
        x = d.loc[a:b, "vor"].notna()
        if x.sum() == 0:
            continue
        pro_tag = x.groupby(x.index.date).sum()
        h_alle.append(pro_tag.median())
        h_mit.append(pro_tag[pro_tag > 0].median())
        anteil.append((pro_tag > 0).mean())
    zeilen.append({"phase": name,
                   "Stunden pro Tag (Median)": round(float(np.median(h_alle)), 1),
                   "Stunden pro Tag mit Daten": round(float(np.median(h_mit)), 1),
                   "Anteil Tage mit Daten": round(float(np.mean(anteil)), 2)})
display(pd.DataFrame(zeilen))

In [ ]:
G = nx.DiGraph()
G.add_edges_from(zip(ek["von_key"], ek["nach_key"]))
U = G.to_undirected()
unbek = [n for n in G if n not in koord]

# Positionen: bekannte übernehmen, unbekannte in die Mitte ihrer Nachbarn setzen
pos = dict(koord)
for comp in nx.connected_components(U):
    bekannt = [pos[n] for n in comp if n in koord]
    if not bekannt:
        continue
    start = tuple(np.mean(bekannt, axis=0))
    for n in comp:
        pos.setdefault(n, start)
for _ in range(300):
    for n in unbek:
        if n in pos:
            nb = [pos[m] for m in U[n] if m in pos]
            if nb:
                pos[n] = tuple(np.mean(nb, axis=0))
geschaetzt = [n for n in unbek if n in pos]

print(f"Nodes: {G.number_of_nodes()} | mit echter Position: {len(koord)} | geschätzt: {len(geschaetzt)} "
      f"| ohne Position: {len(unbek) - len(geschaetzt)}")
print("pos ist:", type(pos).__name__, "mit", len(pos), "Einträgen")

In [ ]:
VMIN, VMAX = 0, 50            # Farbskala der Spreizung in K, nach deinen Daten anpassen
ZEITRAUM = (None, None)       # z.B. ("2026-01-01", "2026-03-31") für die Karte und das Ranking

adr = tab.drop_duplicates("zaehler").set_index("zaehler")["adresse"].to_dict()
t = tag.loc[ZEITRAUM[0]:ZEITRAUM[1]]
reihenfolge = t.median().sort_values().index

# 1) Heatmap: eine Zeile pro Zähler, eine Spalte pro Tag
H = t[reihenfolge].T
cmap = plt.get_cmap("RdYlGn").copy(); cmap.set_bad("#e5e5e5")
x = mdates.date2num(H.columns.to_pydatetime())
fig, ax = plt.subplots(figsize=(14, max(6, 0.18 * len(H))))
im = ax.imshow(H.values, aspect="auto", cmap=cmap, vmin=VMIN, vmax=VMAX,
               extent=[x[0], x[-1], len(H), 0], interpolation="nearest")
ax.xaxis_date()
ax.set_yticks(np.arange(len(H)) + 0.5)
ax.set_yticklabels([f"{z}  {str(adr.get(z, ''))[:24]}" for z in H.index], fontsize=5)
fig.colorbar(im, ax=ax, label="Spreizung in K (Tagesmedian, grau = keine Daten)")
ax.set_title("Spreizung pro Haus im Zeitverlauf (sortiert: niedrigste Spreizung oben)")
plt.tight_layout(); plt.savefig("spreizung_heatmap.png", dpi=150); plt.show()

# 2) Ein Haus im Detail: Vorlauf, Rücklauf und Spreizung
def zeige(zaehler, von=None, bis=None, frequenz="1h"):
    z = norm_z(zaehler)
    d = serien[z].loc[von:bis, ["vor", "rueck", "spreizung"]].resample(frequenz).mean()
    fig, (a1, a2) = plt.subplots(2, 1, sharex=True, figsize=(13, 6), gridspec_kw={"height_ratios": [2, 1]})
    a1.plot(d.index, d["vor"], color="#d6402d", lw=0.9, label="Vorlauf")
    a1.plot(d.index, d["rueck"], color="#2e6db3", lw=0.9, label="Rücklauf")
    a1.set_ylabel("°C"); a1.legend(loc="upper right"); a1.grid(alpha=0.3)
    a1.set_title(f"Zähler {z} | {adr.get(z, '')}")
    a2.plot(d.index, d["spreizung"], color="black", lw=0.8); a2.axhline(0, color="gray", lw=0.8)
    a2.set_ylabel("Spreizung in K"); a2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as widgets
    dd = widgets.Dropdown(options=[(f"{z}  {adr.get(z, '')}", z) for z in reihenfolge], description="Zähler")
    widgets.interact(lambda zaehler: zeige(zaehler), zaehler=dd)
except ImportError:
    print("Auswahlfeld: %pip install ipywidgets. Ohne geht auch:  zeige('Zählernummer')")

# 3) Ranking
rang = pd.DataFrame({"adresse": [adr.get(z, "") for z in t.columns],
                     "spreizung_median_K": t.median().round(1).values,
                     "spreizung_p10_K": t.quantile(0.1).round(1).values,
                     "spreizung_p90_K": t.quantile(0.9).round(1).values,
                     "tage_mit_daten": t.notna().sum().values}, index=t.columns).sort_values("spreizung_median_K")
display(rang.head(15))
rang.to_csv("spreizung_ranking.csv", sep=";", encoding="utf-8-sig")

# 4) Karte: Punkte nach mittlerer Spreizung gefärbt
med = t.median()
kn = knoten.copy()
kn["spreizung"] = kn["zaehler"].apply(lambda s: pd.Series([med.get(norm_z(z), np.nan) for z in s.split(", ")]).mean())
cm = branca.colormap.LinearColormap(["#d73027", "#fee08b", "#1a9850"], vmin=VMIN, vmax=VMAX,
                                    caption="Spreizung in K (Median im gewählten Zeitraum)")
karte_s = folium.Map(location=[kn["lat"].mean(), kn["lon"].mean()], zoom_start=17, tiles="OpenStreetMap")
if "G" in globals() and isinstance(globals().get("pos"), dict):      # Kanten als Hintergrund, falls vorhanden
    for v, w in G.edges:
        if v in pos and w in pos:
            folium.PolyLine([pos[v], pos[w]], color="#999999", weight=2, opacity=0.6).add_to(karte_s)
for r in kn.itertuples():
    farbe = "#888888" if pd.isna(r.spreizung) else cm(r.spreizung)
    folium.CircleMarker([r.lat, r.lon], radius=7 + 2 * min(r.anzahl - 1, 4), color="black", weight=1,
                        fill=True, fill_color=farbe, fill_opacity=0.95,
                        tooltip=f"{r.adresse} | Spreizung: {'keine Daten' if pd.isna(r.spreizung) else f'{r.spreizung:.1f} K'} | Zähler: {r.zaehler}").add_to(karte_s)
cm.add_to(karte_s)
b = kn[["lat", "lon"]].agg(["min", "max"])
karte_s.fit_bounds([[b.loc["min", "lat"], b.loc["min", "lon"]], [b.loc["max", "lat"], b.loc["max", "lon"]]])
karte_s.save("karte_spreizung.html")
display(karte_s)

In [ ]:
exec(open(ROOT / "notebooks" / "dashboard_monitor.py", encoding="utf-8").read(), globals())

## Monitorstatus auf der Karte

- Standardfärbung: Priorität aus Notebook 22 im Winter des gewählten Zeitpunkts (Mai bis September: letzter Winter), bei mehreren Zählern je Knoten der schlechteste.
- Zweite Färbung: Alarmbeginn laut CUSUM höchstens 14 Tage vor dem Zeitpunkt. Orange: alle Verschlechterungen in einer Netzereignis-Episode, Hinweis auf Wetter oder Netz.
- "Ganze Analyse anzeigen" öffnet das Stationsblatt (`src/stationsblatt.py`) mit Carpet und Dauerlinie aus Notebook 13. Bilder relativ verlinkt, die HTML-Datei muss in `outputs/netz/` bleiben.
- Die folgende Zelle zählt nur Zählernummern und Kategorien.

In [ ]:
import pandas as pd

zeilen = []
for nj in nodes_js:
    for w, s in nj.get("mon", {}).get("w", {}).items():
        zeilen.append({"winter": w, "prioritaet": ["kein", "niedrig", "mittel", "hoch"][s["p"]]})
auf_karte = (pd.DataFrame(zeilen).value_counts(["winter", "prioritaet"]).unstack(fill_value=0)
             .reindex(columns=["hoch", "mittel", "niedrig"], fill_value=0))
print("Häuser auf der Karte je Winter und Priorität")
print(auf_karte.to_string())

status = pd.read_csv(ROOT / "outputs" / "monitor_status.csv", dtype={"meter": str})
karte_z = {z for nj in nodes_js for z in nj["z"].split(", ") if z}
fehlt = sorted(set(status["meter"].map(norm_z)) - karte_z)
print(f"\nZähler im Monitor: {status['meter'].nunique()}, davon auf der Karte: "
      f"{status['meter'].nunique() - len(fehlt)}")
print("Nicht auf der Karte (keine Kante oder keine Position):", ", ".join(fehlt) or "keine")
print("Netzereignis-Episoden:", len(ereignisse_js))
mit_blatt = [nj for nj in nodes_js if nj.get("bl")]
ohne_blatt = sorted(z for nj in nodes_js if nj["m"] >= 0 and not nj.get("bl") for z in nj["z"].split(", "))
print(f"Ganze Analyse verfügbar: {len(mit_blatt)} Häuser, ohne Stationsblatt: {', '.join(ohne_blatt) or 'keins'}")

## Zusammenfassung & Bewertung

- Belastbar: Monitorstatus und Alarme hängen über die Zählernummer an den Knoten aus Notebook 16, die Karte zeigt Lage der Stationen mit hoher Priorität und Netzereignis-Bezug.
- Schwach: Nachbarn alarmieren nicht häufiger gemeinsam als zufällige Paare (8 von 9 Tests mit p > 0,05), räumliche Häufungen belegen keinen Netzfehler. Zähler ohne Kante oder Position fehlen.
- Kartenzeitraum Januar 2024 bis Mai 2025, Winter 22/23 nicht sichtbar.
- Offen: Zeitraum erweitern, Monitorebene in Notebook 16 übernehmen oder Kopie behalten.